# Phase 2D -- codec-robust voice head (Model F2)

Implements `docs/PHASE_2D_CODEC_ROBUSTNESS.md` and its addendum. Generated from
`notebooks/src/06_voice_codec_robust.py` by `scripts/build_notebooks.py` -- edit the source, not this .ipynb.
The criteria this notebook is judged by are written in `docs/design_decisions.md` ("Phase 2D") and were fixed
before any run.

**Question.** Compressed audio (AAC, MP3, Opus, phone) pushes genuine speech towards "fake" for the served
voice model. Does training the same small head on features of codec-processed copies of the training clips
fix that, without hurting clean audio?

**What this notebook does**
1. Rebuilds exactly the speaker-disjoint P1 split Model D used (and checks it against the recorded hash).
2. Turns every clip into pooled XLS-R features: the clean clip, 3 augmented copies of each training clip, and
   every codec condition of the test clips. Real codecs through ffmpeg, one clip at a time, nothing but
   feature vectors stored, in chunks, so a crash or the 12 h session limit loses little.
3. Trains a Model D control (clean features only) and F2 (clean + augmented) with the same recipe, 3 seeds.
4. Evaluates both on every condition with paired bootstrap intervals, and writes aggregates only.

**Two switches, in the cell right below.** `RUN_MODE = "PILOT"` measures speed and size on about 500 clips and
stops. Change it to `"FULL"` for the real run. Read `docs/KAGGLE_RUNBOOK_PHASE_2D.md` first.

Nothing here changes the served model. Heads written by this notebook stay private until the owner has checked
the dataset terms.

## Setup

In [ ]:
# Kaggle: Accelerator = GPU, Internet = on (needed to download the dataset and the encoder).
import os
import shutil
import subprocess
import sys

# True only when tests/smoke_notebook_06.py runs this notebook's code on synthetic data.
SMOKE_TEST = os.environ.get("TRUSTMESH_SMOKE") == "1"

# ---- the switches --------------------------------------------------------------------------------------------
RUN_MODE = os.environ.get("TRUSTMESH_RUN_MODE", "PILOT")   # "PILOT" (measure and stop) or "FULL"
PILOT_CLIPS = 24 if SMOKE_TEST else 500                    # clips the pilot processes
EVAL_PER_CONDITION = 60 if SMOKE_TEST else 4000            # test clips per condition (all bonafide + spoof up to this); None = all
MAX_EXTRACT_HOURS = 9.0                                    # extraction stops cleanly after this and says how to continue
# Directories holding chunk files from an earlier session (e.g. an attached notebook output). Leave empty on a first run.
RESUME_DIRS = [d for d in os.environ.get("TRUSTMESH_RESUME_DIRS", "").split(os.pathsep) if d]
assert RUN_MODE in ("PILOT", "FULL"), "RUN_MODE must be 'PILOT' or 'FULL'"


def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


if not SMOKE_TEST:
    for module_name, package in [("datasets", "datasets"), ("soundfile", "soundfile"),
                                 ("transformers", "transformers"), ("fsspec", "fsspec")]:
        try:
            __import__(module_name)
        except ImportError:
            pip_install(package)
if shutil.which("ffmpeg") is None:
    try:
        __import__("imageio_ffmpeg")
    except ImportError:
        pip_install("imageio-ffmpeg")

In [ ]:
import json
import time
import traceback
from pathlib import Path

import numpy as np
import torch

RUN_START = time.time()
SEED = 42
SEEDS = [42, 43, 44]
torch.manual_seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = Path(os.environ.get("TRUSTMESH_OUTPUT_DIR", "./artifacts/voice"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = Path(os.environ.get("TRUSTMESH_CACHE_DIR", str(OUTPUT_DIR / "cache_phase2d")))
DELETE_CACHE_AT_END = True      # only after a complete, successful FULL run
DEVICE = torch.device("cuda" if torch.cuda.is_available() and not SMOKE_TEST else "cpu")

# Same data and split settings as Phase 2B/2C (notebook 04). Do not change: they define "the same split as Model D".
SPOOF_PER_BONAFIDE = 4
CLIP_SECONDS = 4.0
CLIP_SAMPLES = int(16000 * CLIP_SECONDS)
TEST_FRACTION, VAL_FRACTION = 0.3, 0.2
# Model D's recipe (notebook 04, Phase 2C): kept exactly.
SSL_MODEL = "facebook/wav2vec2-xls-r-300m"
SSL_BATCH = 4 if SMOKE_TEST else 16
SSL_EPOCHS = 3 if SMOKE_TEST else 60
SSL_LR, SSL_PATIENCE = 1e-3, 8
BATCH_SIZE = 8 if SMOKE_TEST else 64
N_BOOTSTRAP = 50 if SMOKE_TEST else 1000
CHUNK_SIZE = 16 if SMOKE_TEST else 256
WORKERS = 2 if SMOKE_TEST else min(4, os.cpu_count() or 2)
PILOT_DIR = CACHE_DIR / "pilot"

# What the real Model D run (results_phase2b.json / phase2c_test_scores.npz) recorded for the P1 split.
EXPECTED_P1 = {"train": {"bonafide": 4099, "spoof": 16099}, "val": {"bonafide": 1091, "spoof": 4329},
               "test": {"bonafide": 2165, "spoof": 8992}}
EXPECTED_TEST_IDX_SHA256 = "5e6a50f79e4cafbc370b28f9569bc9e925449402d761e798588f1f4a4c905f2a"

RESULTS_PATH = OUTPUT_DIR / "results_phase2d.json"
RESULTS = {"smoke_test": SMOKE_TEST, "run_mode": RUN_MODE, "not_run": {}}


def save_progress(key, value):
    RESULTS[key] = value
    RESULTS_PATH.write_text(json.dumps(RESULTS, indent=2, default=float), encoding="utf-8")


print("device:", DEVICE, "| mode:", RUN_MODE, "| SMOKE_TEST:", SMOKE_TEST, "| torch", torch.__version__)

## Shared voice-engine code (inlined from `engines/voice/`)

In [ ]:
# ---- inlined from engines/voice/data.py (edit the module, then rebuild) ----
"""ASVspoof 2019 LA loading and train/test splits.

Dataset: `SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA` on HuggingFace -- parquet, no loading script,
71,237 clips, 16 kHz FLAC. IMPORTANT: it contains only the **evaluation** partition of LA
(7,355 bonafide / 63,882 spoof). The HF copy carrying the official train/dev/eval partitions
(`LanceaKing/asvspoof2019`) needs `trust_remote_code`, which current `datasets` versions no longer
support, so the official protocol (train on attacks A01-A06, test on unseen A07-A19) is not
reachable here. We therefore split the eval partition ourselves; see `speaker_disjoint_split` for
why the split is speaker-disjoint and what that does and does not fix.

Clips are streamed (not downloaded in full) so the subset size is a knob, and audio is fetched
undecoded and decoded here with soundfile -- decoding through the `datasets` Audio feature changed
backends across versions, and doing it ourselves keeps this working either way.
"""
import io
import json

import numpy as np

ASVSPOOF_DATASET = "SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA"
ASVSPOOF_SPLIT = "test"  # the dataset's only split: the LA evaluation partition
IN_THE_WILD_DATASET = "mueller91/In-The-Wild"
TARGET_SR = 16000
BONAFIDE, SPOOF = 1, 0  # label 1 = bonafide = the positive/target class for EER


def _decode_audio(blob):
    """Decode an undecoded `datasets` audio value to (float32 mono waveform, sample rate)."""
    import soundfile as sf

    if isinstance(blob, dict) and blob.get("bytes"):
        wave, sr = sf.read(io.BytesIO(blob["bytes"]), dtype="float32", always_2d=False)
    elif isinstance(blob, dict) and blob.get("array") is not None:
        wave, sr = np.asarray(blob["array"], dtype=np.float32), int(blob["sampling_rate"])
    else:
        path = blob["path"] if isinstance(blob, dict) else blob
        if isinstance(path, str) and "://" in path:
            # Archive-backed datasets (e.g. In-the-Wild, shipped as one zip) come back as chained fsspec
            # URLs like "zip://a.wav::hf://datasets/.../x.zip", which libsndfile can't open. fsspec can,
            # and it only range-reads the needed zip member, not the whole archive.
            import fsspec

            with fsspec.open(path, "rb") as handle:
                wave, sr = sf.read(io.BytesIO(handle.read()), dtype="float32", always_2d=False)
        else:
            wave, sr = sf.read(path, dtype="float32", always_2d=False)
    if wave.ndim > 1:
        wave = wave.mean(axis=1)
    return np.ascontiguousarray(wave, dtype=np.float32), int(sr)


def load_asvspoof_subset(n_clips, seed=42, shuffle_buffer=20000, dataset=ASVSPOOF_DATASET,
                         split=ASVSPOOF_SPLIT, progress_every=500):
    """Stream a shuffled subset of ASVspoof 2019 LA. Returns a list of clip dicts."""
    from datasets import Audio, load_dataset

    stream = load_dataset(dataset, split=split, streaming=True)
    stream = stream.cast_column("audio", Audio(decode=False))
    stream = stream.shuffle(seed=seed, buffer_size=shuffle_buffer).take(n_clips)

    clips = []
    for row in stream:
        wave, sr = _decode_audio(row["audio"])
        try:
            notes = json.loads(row.get("notes") or "{}")
        except (json.JSONDecodeError, TypeError):
            notes = {}
        # The dataset's ClassLabel is 0=bonafide, 1=spoof; we flip so 1 = bonafide = target class.
        clips.append({
            "waveform": wave,
            "sample_rate": sr,
            "label": BONAFIDE if int(row["label"]) == 0 else SPOOF,
            "speaker_id": notes.get("speaker_id") or "unknown",
            "utterance_id": notes.get("utterance_id") or row.get("path") or f"clip_{len(clips)}",
            "path": row.get("path"),
        })
        if progress_every and len(clips) % progress_every == 0:
            print(f"  loaded {len(clips)}/{n_clips} clips")
    return clips


def load_in_the_wild_clips(n_clips, seed=42, shuffle_buffer=2000, dataset=IN_THE_WILD_DATASET):
    """A handful of out-of-domain clips for the generalization spot-check. Column names on this
    dataset aren't documented, so the audio/label columns are detected rather than assumed."""
    from datasets import Audio, load_dataset

    stream = load_dataset(dataset, split="train", streaming=True)
    columns = list(next(iter(stream.take(1))).keys())
    audio_col = next((c for c in columns if "audio" in c.lower() or c.lower() in ("file", "path")), None)
    label_col = next((c for c in columns if c.lower() in ("label", "key", "class", "type", "target")), None)
    if audio_col is None:
        raise RuntimeError(f"no audio-looking column in {columns}")

    try:
        stream = stream.cast_column(audio_col, Audio(decode=False))
    except (ValueError, TypeError):
        pass
    stream = stream.shuffle(seed=seed, buffer_size=shuffle_buffer).take(n_clips)

    clips = []
    for row in stream:
        wave, sr = _decode_audio(row[audio_col])
        raw_label = row.get(label_col)
        label = None
        if isinstance(raw_label, str):
            label = BONAFIDE if raw_label.strip().lower() in ("bona-fide", "bonafide", "real", "genuine") else SPOOF
        elif isinstance(raw_label, (int, np.integer)):
            label = int(raw_label)  # unknown polarity; recorded but flagged in the results
        clips.append({"waveform": wave, "sample_rate": sr, "label": label,
                      "raw_label": raw_label, "label_column": label_col, "speaker_id": "unknown",
                      "utterance_id": f"itw_{len(clips)}", "path": row.get("path")})
    return clips


ASVSPOOF_PROTOCOL = "protocols/ASVspoof2019.LA.cm.eval.trl.txt"
IN_THE_WILD_ARCHIVE = "datasets/mueller91/In-The-Wild/release_in_the_wild.zip"
_BONAFIDE_WORDS = {"bona-fide", "bonafide", "bona fide", "real", "genuine"}
_SPOOF_WORDS = {"spoof", "fake", "deepfake", "synthetic"}


def load_asvspoof_table(dataset=ASVSPOOF_DATASET, split=ASVSPOOF_SPLIT, cache_dir=None):
    """Download the whole parquet once and return it with audio left undecoded.

    Phase 2 streamed a shuffled sample. Phase 2B keeps *every* bonafide clip, which means touching
    every row anyway, and random access by index into a local memory-mapped table beats a streamed
    shuffle buffer for that (streaming, not training, was the slow part of Phase 2).
    """
    from datasets import Audio, load_dataset

    table = load_dataset(dataset, split=split, cache_dir=cache_dir)
    return table.cast_column("audio", Audio(decode=False))


def table_metadata(table):
    """Labels, utterance and speaker IDs for every row, without decoding any audio."""
    labels = np.array([BONAFIDE if int(v) == 0 else SPOOF for v in table["label"]])
    notes = []
    for raw in table["notes"]:
        try:
            notes.append(json.loads(raw or "{}"))
        except (json.JSONDecodeError, TypeError):
            notes.append({})
    paths = list(table["path"])
    utterances = [n.get("utterance_id") or (p.rsplit(".", 1)[0] if p else f"row_{i}")
                  for i, (n, p) in enumerate(zip(notes, paths))]
    return {
        "label": labels,
        "utterance_id": utterances,
        "speaker_id": [n.get("speaker_id") or "unknown" for n in notes],
        "columns": list(table.features),
        "note_keys": sorted({k for n in notes for k in n}),
    }


def decode_row(table, index):
    return _decode_audio(table[int(index)]["audio"])


def load_eval_protocol(path=None, dataset=ASVSPOOF_DATASET, filename=ASVSPOOF_PROTOCOL):
    """Official ASVspoof CM protocol: one line per clip, `SPEAKER UTTERANCE - ATTACK KEY`.

    The parquet has no attack column, but the same HF repo ships this file, which is the source of
    attack IDs (A07-A19). It is only trusted after `verify_protocol` passes.
    """
    if path is None:
        from huggingface_hub import hf_hub_download

        path = hf_hub_download(repo_id=dataset, filename=filename, repo_type="dataset")
    protocol = {}
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            parts = line.split()
            if len(parts) != 5:
                continue
            speaker, utterance, _, attack, key = parts
            protocol[utterance] = {
                "speaker_id": speaker,
                "attack": "bonafide" if attack == "-" else attack,
                "label": BONAFIDE if key == "bonafide" else SPOOF,
            }
    return protocol, str(path)


def verify_protocol(protocol, utterance_ids, labels, speaker_ids):
    """The join is only usable if it reproduces the dataset's own labels and speakers exactly --
    a protocol file for a different partition, or misaligned IDs, fails this loudly."""
    n = len(utterance_ids)
    found = label_ok = speaker_ok = 0
    bonafide_clean = True
    for utterance, label, speaker in zip(utterance_ids, labels, speaker_ids):
        entry = protocol.get(utterance)
        if entry is None:
            continue
        found += 1
        label_ok += entry["label"] == label
        speaker_ok += entry["speaker_id"] == speaker
        if label == BONAFIDE and entry["attack"] != "bonafide":
            bonafide_clean = False
    return {
        "clips": n,
        "found_in_protocol": found,
        "label_agreement": label_ok,
        "speaker_agreement": speaker_ok,
        "bonafide_rows_have_no_attack": bonafide_clean,
        "usable": found == n and label_ok == n and speaker_ok == n and bonafide_clean,
    }


def select_subset(labels, spoof_per_bonafide, seed=42):
    """Every bonafide clip plus a random sample of spoof clips capped at `spoof_per_bonafide` per
    bonafide clip. If a budget forces a smaller set, lower the ratio -- bonafide is never cut."""
    labels = np.asarray(labels)
    bonafide = np.flatnonzero(labels == BONAFIDE)
    spoof = np.flatnonzero(labels == SPOOF)
    n_spoof = min(len(spoof), int(round(spoof_per_bonafide * len(bonafide))))
    rng = np.random.default_rng(seed)
    return np.sort(np.concatenate([bonafide, rng.choice(spoof, n_spoof, replace=False)]))


def choose_held_out_attacks(attacks, n_held_out, seed=42):
    spoof_attacks = sorted({a for a in attacks if a != "bonafide"})
    n = max(1, min(n_held_out, len(spoof_attacks) - 1))  # leave at least one attack for training
    rng = np.random.default_rng(seed)
    return sorted(rng.choice(spoof_attacks, size=n, replace=False).tolist())


def drop_attacks(indices, attacks, held_out):
    """Train/val side of the attack-disjoint protocol: bonafide plus spoofs from non-held-out attacks."""
    held = set(held_out)
    return [i for i in indices if attacks[i] == "bonafide" or attacks[i] not in held]


def keep_only_attacks(indices, attacks, held_out):
    """Test side of the attack-disjoint protocol: bonafide plus spoofs from held-out attacks only."""
    held = set(held_out)
    return [i for i in indices if attacks[i] == "bonafide" or attacks[i] in held]


class ZipAudioArchive:
    """Members of a zip that may be remote (hf://, https://), read without downloading it whole.

    One zip filesystem is opened and reused, so the central directory of a multi-GB archive is read
    once and each clip is a single range read -- reopening a chained `zip://...::hf://...` URL per clip
    (what Phase 2's loader effectively did) re-reads the directory every time.
    """

    def __init__(self, archive, protocol="hf"):
        import fsspec

        self.archive, self.protocol = archive, protocol
        self.fs = fsspec.filesystem("zip", fo=archive, target_protocol=protocol)

    def members(self):
        return sorted(self.fs.find(""))

    def read_bytes(self, member):
        with self.fs.open(member, "rb") as handle:
            return handle.read()


def read_archive_metadata(archive):
    """Find the label metadata inside the archive (In-the-Wild ships `meta.csv` with file/label
    columns) rather than assuming a HuggingFace column schema that isn't documented."""
    import pandas as pd

    members = archive.members()
    csvs = [m for m in members if m.lower().endswith(".csv")]
    if not csvs:
        raise RuntimeError(f"no metadata .csv among {len(members)} archive members")
    meta_member = next((m for m in csvs if m.rsplit("/", 1)[-1].lower() == "meta.csv"), csvs[0])
    df = pd.read_csv(io.BytesIO(archive.read_bytes(meta_member)))

    def looks_like_files(col):
        return df[col].astype(str).str.lower().str.endswith((".wav", ".flac", ".mp3")).mean() > 0.9

    def looks_like_labels(col):
        values = set(df[col].astype(str).str.strip().str.lower().unique())
        return bool(values) and values <= (_BONAFIDE_WORDS | _SPOOF_WORDS)

    file_col = next((c for c in df.columns if looks_like_files(c)), None)
    label_col = next((c for c in df.columns if looks_like_labels(c)), None)
    if file_col is None or label_col is None:
        raise RuntimeError(f"could not identify file/label columns in {meta_member}: {list(df.columns)}")

    prefix = meta_member.rsplit("/", 1)[0] + "/" if "/" in meta_member else ""
    raw = df[label_col].astype(str).str.strip()
    table = pd.DataFrame({
        "member": prefix + df[file_col].astype(str),
        "raw_label": raw,
        "label": np.where(raw.str.lower().isin(_BONAFIDE_WORDS), BONAFIDE, SPOOF),
    })
    info = {"metadata_member": meta_member, "file_column": file_col, "label_column": label_col,
            "raw_label_values": sorted(raw.unique().tolist()), "rows": int(len(table)),
            "rows_bonafide": int((table.label == BONAFIDE).sum()), "rows_spoof": int((table.label == SPOOF).sum())}
    return table, info


def load_archive_clips_balanced(archive, n_per_class, seed=42, max_failures=None):
    """`n_per_class` clips of each class, read one by one; every failure is recorded, not hidden."""
    import soundfile as sf

    table, info = read_archive_metadata(archive)
    max_failures = max_failures if max_failures is not None else 2 * n_per_class
    rng = np.random.default_rng(seed)
    clips, failures = [], []
    for label in (BONAFIDE, SPOOF):
        pool = table[table.label == label].reset_index(drop=True)
        loaded = 0
        for pos in rng.permutation(len(pool)):
            if loaded >= n_per_class:
                break
            row = pool.iloc[int(pos)]
            try:
                wave, sr = sf.read(io.BytesIO(archive.read_bytes(row.member)), dtype="float32", always_2d=False)
            except Exception as exc:  # noqa: BLE001 -- recorded per clip, and capped below
                failures.append({"member": row.member, "error": f"{type(exc).__name__}: {exc}"})
                if len(failures) > max_failures:
                    raise RuntimeError(f"{len(failures)} clips failed to load; first: {failures[0]}") from exc
                continue
            if wave.ndim > 1:
                wave = wave.mean(axis=1)
            clips.append({"waveform": np.ascontiguousarray(wave, dtype=np.float32), "sample_rate": int(sr),
                          "label": label, "raw_label": row.raw_label, "speaker_id": "unknown",
                          "utterance_id": row.member, "attack": "bonafide" if label == BONAFIDE else "unknown"})
            loaded += 1
    report = {**info, "requested_per_class": n_per_class, "loaded": len(clips),
              "loaded_bonafide": sum(c["label"] == BONAFIDE for c in clips),
              "loaded_spoof": sum(c["label"] == SPOOF for c in clips),
              "failed": len(failures), "failures": failures[:20]}
    return clips, report


def write_synthetic_archive(path, n_real=4, n_fake=5, include_broken=True, sample_rate=TARGET_SR):
    """In-the-Wild-shaped zip (release_in_the_wild/*.wav + meta.csv with file/speaker/label) for the
    local smoke test and unit tests -- exercises the same fsspec zip path as the real hf:// archive.
    `include_broken` adds one undecodable member so failure recording is exercised too."""
    import soundfile as sf
    import zipfile

    rows = ["file,speaker,label"]
    rng = np.random.default_rng(0)
    with zipfile.ZipFile(path, "w") as zf:
        for i in range(n_real + n_fake):
            label = "bona-fide" if i < n_real else "spoof"
            t = np.arange(int(1.5 * sample_rate)) / sample_rate
            wave = (0.2 * np.sin(2 * np.pi * rng.uniform(100, 300) * t)).astype(np.float32)
            buf = io.BytesIO()
            sf.write(buf, wave, sample_rate, format="WAV")
            zf.writestr(f"release_in_the_wild/{i}.wav", buf.getvalue())
            rows.append(f"{i}.wav,Speaker {i % 3},{label}")
        if include_broken:
            zf.writestr("release_in_the_wild/broken.wav", b"not audio")
            rows.append("broken.wav,Speaker 0,spoof")
        zf.writestr("release_in_the_wild/meta.csv", "\n".join(rows))
    return str(path)


def write_synthetic_protocol(clips, path):
    """Protocol file in the official format, for the smoke test's protocol-verification path."""
    with open(path, "w", encoding="utf-8") as handle:
        for clip in clips:
            attack = "-" if clip["label"] == BONAFIDE else clip["attack"]
            key = "bonafide" if clip["label"] == BONAFIDE else "spoof"
            handle.write(f"{clip['speaker_id']} {clip['utterance_id']} - {attack} {key}\n")
    return str(path)


def make_synthetic_asvspoof(n_clips=120, seed=0, sample_rate=TARGET_SR, n_attacks=6, bonafide_pad_s=(0.02, 0.1)):
    """Tiny ASVspoof-shaped set for local smoke tests only -- never used for reported numbers.

    Deliberately includes the failure mode the real data is suspected of: spoof clips get more
    leading/trailing silence than bonafide ones, so the silence-leakage check has something to find.
    Spoof clips get one of `n_attacks` synthetic attack IDs, drawn independently of speaker (as in real
    ASVspoof, where every target speaker is spoofed by every system), from a separate RNG so adding
    attacks didn't change the synthetic audio other tests rely on.
    """
    rng = np.random.default_rng(seed)
    attack_rng = np.random.default_rng(seed + 1)
    clips = []
    for i in range(n_clips):
        label = BONAFIDE if i % 4 == 0 else SPOOF  # ~25% bonafide, imbalanced like LA
        attack = "bonafide" if label == BONAFIDE else f"A{7 + int(attack_rng.integers(n_attacks)):02d}"
        speech_s = rng.uniform(1.0, 2.5)
        t = np.arange(int(speech_s * sample_rate), dtype=np.float32) / sample_rate
        f0 = rng.uniform(100, 200)
        speech = 0.3 * np.sin(2 * np.pi * f0 * t) + 0.1 * np.sin(2 * np.pi * 2 * f0 * t)
        if label == SPOOF:  # a crude high-frequency "vocoder artifact"
            speech += 0.05 * np.sin(2 * np.pi * 6500 * t)
        speech += 0.01 * rng.standard_normal(speech.shape).astype(np.float32)
        pad_s = rng.uniform(0.4, 0.8) if label == SPOOF else rng.uniform(*bonafide_pad_s)
        pad = 1e-4 * rng.standard_normal(int(pad_s * sample_rate)).astype(np.float32)
        clips.append({
            "waveform": np.concatenate([pad, speech, pad]).astype(np.float32),
            "sample_rate": sample_rate,
            "label": label,
            "speaker_id": f"SPK_{i % 12:02d}",
            "utterance_id": f"SYN_{i:04d}",
            "path": f"synthetic/{i}.wav",
            "attack": attack,
        })
    return clips


def class_counts(clips):
    labels = np.array([c["label"] for c in clips])
    return {"bonafide": int((labels == BONAFIDE).sum()), "spoof": int((labels == SPOOF).sum()),
            "total": int(len(labels))}


def pos_weight(clips):
    """Weight for the positive (bonafide, minority) class in BCE, as in the graph engine."""
    counts = class_counts(clips)
    return float(counts["spoof"] / max(counts["bonafide"], 1))


def _both_classes_present(clips, indices):
    labels = {clips[i]["label"] for i in indices}
    return BONAFIDE in labels and SPOOF in labels


def speaker_disjoint_split(clips, test_fraction=0.3, seed=42, attempts=12):
    """Split so no speaker appears in both train and test.

    A random clip-level split lets the same speaker land on both sides, and a model can then key on
    speaker identity/recording channel instead of spoofing artifacts -- the audio analogue of the
    Phase 1 random-vs-temporal split leak. Speakers are shuffled and sent whole to the test side
    until it reaches `test_fraction`, so a speaker is never split across sides.

    Caveat this does NOT fix: every clip here comes from the LA *evaluation* partition, so the same
    spoofing systems (A07-A19) appear on both sides. Unseen-attack generalization is not measured.
    """
    by_speaker = {}
    for idx, clip in enumerate(clips):
        by_speaker.setdefault(clip["speaker_id"], []).append(idx)

    target_test = test_fraction * len(clips)
    for attempt in range(attempts):
        rng = np.random.default_rng(seed + attempt)
        speakers = sorted(by_speaker)
        rng.shuffle(speakers)

        train_idx, test_idx, n_test = [], [], 0
        for speaker in speakers:
            members = by_speaker[speaker]
            # Send speakers to test until adding another would overshoot the target by more than it undershoots.
            if n_test + len(members) / 2 <= target_test:
                test_idx.extend(members)
                n_test += len(members)
            else:
                train_idx.extend(members)
        # Both sides need both classes, or EER is undefined on that side. Speakers are assigned whole,
        # so this isn't guaranteed by construction -- reshuffle and retry.
        if train_idx and test_idx and _both_classes_present(clips, train_idx) and _both_classes_present(clips, test_idx):
            return sorted(train_idx), sorted(test_idx)

    # Too few speakers (or a speaker holds an entire class) for a disjoint split with both classes on
    # both sides. Fall back to a random split rather than returning an unusable one; the caller's
    # split report will show shared_speakers > 0, making the fallback visible instead of silent.
    return random_split(clips, test_fraction, seed)


def random_split(clips, test_fraction=0.3, seed=42):
    """Clip-level random split -- speakers appear on both sides. Kept as the leakage comparison."""
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(clips))
    n_test = int(round(test_fraction * len(clips)))
    return sorted(order[n_test:].tolist()), sorted(order[:n_test].tolist())


def split_report(clips, train_idx, test_idx):
    train_speakers = {clips[i]["speaker_id"] for i in train_idx}
    test_speakers = {clips[i]["speaker_id"] for i in test_idx}
    return {
        "train": class_counts([clips[i] for i in train_idx]),
        "test": class_counts([clips[i] for i in test_idx]),
        "train_speakers": len(train_speakers),
        "test_speakers": len(test_speakers),
        "shared_speakers": len(train_speakers & test_speakers),
    }

In [ ]:
# ---- inlined from engines/voice/preprocess.py (edit the module, then rebuild) ----
"""Audio front-end: resample -> silence trim -> fixed length -> LFCC (+deltas), plus the silence
ablations and training-time augmentation used in Phase 2B.

**Why LFCC, not mel.** LFCC is the front-end of the official ASVspoof 2019/2021 baselines. Its
filters are spaced linearly, so it keeps resolution at high frequencies, which is where vocoder and
TTS artifacts live. Mel spacing compresses exactly that region because it is tuned to human
perception, which is the right trade for ASR and the wrong one for spoof detection.

LFCC is implemented here in plain torch (linear triangular filterbank -> log -> DCT-II -> deltas)
rather than via `torchaudio.transforms.LFCC`, so the pipeline has no torchaudio dependency.
`LFCCExtractor` is batched and runs on any device; `VoicePreprocessor` uses the same extractor for
single clips, so cached features and on-the-fly (augmented) features are the same computation.
"""
import math

import numpy as np
import torch
import torch.nn.functional as F

TARGET_SR = 16000


def resample_linear(waveform, sr, target_sr=TARGET_SR):
    """Linear-interpolation resample. ASVspoof LA is already 16 kHz, so this is normally a no-op;
    it exists so an odd-rate out-of-domain clip doesn't silently go through at the wrong rate."""
    if sr == target_sr:
        return waveform, sr
    n_out = int(round(len(waveform) * target_sr / sr))
    x_in = np.linspace(0.0, 1.0, num=len(waveform), endpoint=False)
    x_out = np.linspace(0.0, 1.0, num=n_out, endpoint=False)
    return np.interp(x_out, x_in, waveform).astype(np.float32), target_sr


def frame_energy_db(waveform, sr, frame_ms=20.0, hop_ms=10.0):
    frame, hop = int(sr * frame_ms / 1000), int(sr * hop_ms / 1000)
    if len(waveform) < frame:
        waveform = np.pad(waveform, (0, frame - len(waveform)))
    frames = np.lib.stride_tricks.sliding_window_view(waveform, frame)[::hop]
    rms = np.sqrt(np.mean(frames.astype(np.float64) ** 2, axis=1) + 1e-12)
    return 20.0 * np.log10(rms + 1e-12), frame, hop


def trim_silence(waveform, sr, top_db=30.0, frame_ms=20.0, hop_ms=10.0):
    """Energy-based leading/trailing silence trim (librosa.effects.trim-style, no dependency).

    Returns (trimmed waveform, stats). Stats carry the leading/trailing silence and speech durations
    (what the silence-leakage probe measures) and the speech boundaries in samples (what the silence
    ablations cut at).
    """
    db, frame, hop = frame_energy_db(waveform, sr, frame_ms, hop_ms)
    voiced = np.flatnonzero(db > db.max() - top_db)
    total_s = len(waveform) / sr
    if len(voiced) == 0:  # nothing above the threshold: keep the clip, report it as all silence
        return waveform, {"total_s": total_s, "lead_silence_s": 0.0, "trail_silence_s": 0.0,
                          "speech_s": 0.0, "silence_ratio": 1.0, "all_silence": True,
                          "start_sample": 0, "end_sample": len(waveform)}
    start = int(voiced[0] * hop)
    end = min(len(waveform), int(voiced[-1] * hop + frame))
    trimmed = waveform[start:end]
    lead_s, trail_s = start / sr, (len(waveform) - end) / sr
    return trimmed, {
        "total_s": total_s,
        "lead_silence_s": lead_s,
        "trail_silence_s": trail_s,
        "speech_s": len(trimmed) / sr,
        "silence_ratio": (lead_s + trail_s) / total_s if total_s else 0.0,
        "all_silence": False,
        "start_sample": start,
        "end_sample": end,
    }


def fix_length(waveform, target_len):
    """Crop or repeat-pad to a fixed sample count (repeat-padding is the usual ASVspoof practice --
    zero-padding would inject exactly the silence cue this pipeline is trying to remove)."""
    if len(waveform) == 0:
        return np.zeros(target_len, dtype=np.float32)
    if len(waveform) >= target_len:
        return waveform[:target_len]
    repeats = int(np.ceil(target_len / len(waveform)))
    return np.tile(waveform, repeats)[:target_len].astype(np.float32)


# ---------------------------------------------------------------------------------------------------
# Silence ablations (Phase 2B, Task 4). Each variant removes a different part of what the silence
# could be leaking, so comparing them says *which* part the untrimmed model was using:
#
#   variant                    silence duration   silence content (what's actually in it)
#   untrimmed                        kept              kept
#   zeros_original_duration          kept              removed (digital zeros)
#   zeros_fixed / noise_fixed        removed           removed (identical zeros / identical-level noise)
#   silence_only (fixed window)      removed           kept -- and nothing else is kept
#   trimmed                          removed           removed
# ---------------------------------------------------------------------------------------------------
SILENCE_ABLATIONS = ("zeros_fixed", "noise_fixed", "zeros_original_duration")


def ablate_silence(waveform, sr, mode, stats, fixed_seconds=0.5, noise_dbfs=-60.0, rng=None):
    """Rebuild a clip with its leading and trailing silence replaced according to `mode`.

    Both ends are replaced, not just the leading one: trailing silence also leaks the label (it is
    the second-strongest timing cue), so replacing only the lead would leave a known cue in place and
    confound the test.
    """
    start, end = stats["start_sample"], stats["end_sample"]
    speech = waveform[start:end]
    if mode == "zeros_fixed":
        pad = np.zeros(int(fixed_seconds * sr), dtype=np.float32)
        return np.concatenate([pad, speech, pad])
    if mode == "noise_fixed":
        n = int(fixed_seconds * sr)
        rng = rng if rng is not None else np.random.default_rng(0)
        noise = (rng.standard_normal(2 * n) * 10 ** (noise_dbfs / 20)).astype(np.float32)
        return np.concatenate([noise[:n], speech, noise[n:]])
    if mode == "zeros_original_duration":
        return np.concatenate([np.zeros(start, dtype=np.float32), speech,
                               np.zeros(len(waveform) - end, dtype=np.float32)])
    raise ValueError(f"unknown silence ablation: {mode}")


def silence_window(waveform, stats, n_samples):
    """Exactly `n_samples` of the clip's genuine silence (leading, then trailing), or None if the clip
    has less silence than that. A fixed-size window means silence *duration* cannot reach the model;
    only what the silence contains can."""
    start, end = stats["start_sample"], stats["end_sample"]
    silence = np.concatenate([waveform[:start], waveform[end:]])
    if len(silence) < n_samples:
        return None
    return silence[:n_samples].astype(np.float32)


# ---------------------------------------------------------------------------------------------------
# LFCC
# ---------------------------------------------------------------------------------------------------
def linear_filterbank(n_filters, n_fft, sr):
    """Triangular filters spaced linearly in Hz (the 'linear' in LFCC)."""
    edges = torch.linspace(0.0, sr / 2.0, n_filters + 2)
    bin_freqs = torch.linspace(0.0, sr / 2.0, n_fft // 2 + 1)
    bank = torch.zeros(n_filters, len(bin_freqs))
    for i in range(n_filters):
        low, center, high = edges[i], edges[i + 1], edges[i + 2]
        rising = (bin_freqs - low) / torch.clamp(center - low, min=1e-6)
        falling = (high - bin_freqs) / torch.clamp(high - center, min=1e-6)
        bank[i] = torch.clamp(torch.minimum(rising, falling), min=0.0)
    return bank


def dct_matrix(n_out, n_in):
    """Orthonormal DCT-II."""
    n = torch.arange(n_in, dtype=torch.float32)
    k = torch.arange(n_out, dtype=torch.float32).unsqueeze(1)
    mat = torch.cos(torch.pi * k * (2 * n + 1) / (2 * n_in)) * np.sqrt(2.0 / n_in)
    mat[0] = mat[0] / np.sqrt(2.0)
    return mat


def deltas(features, width=2):
    """Standard regression deltas over a +/-`width` frame window, edges replicated. Accepts
    [coeffs, frames] or [batch, coeffs, frames]."""
    squeeze = features.dim() == 2
    x = features.unsqueeze(0) if squeeze else features
    padded = F.pad(x, (width, width), mode="replicate")
    n_frames = x.shape[-1]
    out = torch.zeros_like(x)
    for n in range(1, width + 1):
        out += n * (padded[..., width + n: width + n + n_frames] - padded[..., width - n: width - n + n_frames])
    out = out / (2.0 * sum(n * n for n in range(1, width + 1)))
    return out.squeeze(0) if squeeze else out


class LFCCExtractor(torch.nn.Module):
    """Batched LFCC (+deltas): [batch, samples] -> [batch, n_coeffs*3, frames], on any device.

    `filterbank_augment`, if given, is applied to the log linear-filterbank energies before the DCT.
    That is where frequency masking means something (masking whole frequency bands); masking rows
    of the cepstrum afterwards would not correspond to any frequency region.
    """

    def __init__(self, sample_rate=TARGET_SR, n_fft=512, win_ms=20.0, hop_ms=10.0, n_filters=70,
                 n_coeffs=20, add_deltas=True):
        super().__init__()
        self.n_fft = n_fft
        self.win_length = int(sample_rate * win_ms / 1000)
        self.hop_length = int(sample_rate * hop_ms / 1000)
        self.n_coeffs = n_coeffs
        self.add_deltas = add_deltas
        self.register_buffer("filterbank", linear_filterbank(n_filters, n_fft, sample_rate))
        self.register_buffer("dct", dct_matrix(n_coeffs, n_filters))
        self.register_buffer("window", torch.hann_window(self.win_length))

    @property
    def n_features(self):
        return self.n_coeffs * (3 if self.add_deltas else 1)

    def n_frames(self, n_samples):
        return n_samples // self.hop_length + 1

    def forward(self, waves, filterbank_augment=None):
        spec = torch.stft(waves, n_fft=self.n_fft, hop_length=self.hop_length, win_length=self.win_length,
                          window=self.window, center=True, pad_mode="reflect", return_complex=True)
        log_fbank = torch.log(torch.matmul(self.filterbank, spec.abs() ** 2) + 1e-10)
        if filterbank_augment is not None:
            log_fbank = filterbank_augment(log_fbank)
        feats = torch.matmul(self.dct, log_fbank)
        if self.add_deltas:
            d1 = deltas(feats)
            feats = torch.cat([feats, d1, deltas(d1)], dim=1)
        return feats


class VoicePreprocessor:
    """waveform -> (n_coeffs*3, n_frames) LFCC+delta+delta-delta tensor.

    Defaults follow the ASVspoof LFCC baseline: 20 ms window / 10 ms hop, 70 linear filters,
    20 cepstral coefficients, plus first and second deltas = 60 feature dimensions.
    """

    def __init__(self, sample_rate=TARGET_SR, clip_seconds=4.0, n_fft=512, win_ms=20.0, hop_ms=10.0,
                 n_filters=70, n_coeffs=20, trim=True, top_db=30.0, add_deltas=True):
        self.sample_rate = sample_rate
        self.clip_seconds = clip_seconds
        self.clip_samples = int(sample_rate * clip_seconds)
        self.trim = trim
        self.top_db = top_db
        self.extractor = LFCCExtractor(sample_rate, n_fft, win_ms, hop_ms, n_filters, n_coeffs, add_deltas)
        self.n_coeffs = n_coeffs
        self.add_deltas = add_deltas
        self.hop_length = self.extractor.hop_length

    @property
    def n_features(self):
        return self.extractor.n_features

    @property
    def n_frames(self):
        return self.extractor.n_frames(self.clip_samples)

    def prepare_waveform(self, waveform, sr):
        waveform, sr = resample_linear(np.asarray(waveform, dtype=np.float32), sr, self.sample_rate)
        stats = {"total_s": len(waveform) / sr, "lead_silence_s": 0.0, "trail_silence_s": 0.0,
                 "speech_s": len(waveform) / sr, "silence_ratio": 0.0, "all_silence": False,
                 "start_sample": 0, "end_sample": len(waveform)}
        if self.trim:
            waveform, stats = trim_silence(waveform, sr, self.top_db)
        return fix_length(waveform, self.clip_samples), stats

    @torch.no_grad()
    def lfcc(self, waveform):
        wave = torch.as_tensor(waveform, dtype=torch.float32).unsqueeze(0)
        return self.extractor(wave)[0]

    def __call__(self, waveform, sr):
        fixed, stats = self.prepare_waveform(waveform, sr)
        return self.lfcc(fixed), stats

    def featurize_clips(self, clips, progress_every=1000):
        """Returns (features [N,1,F,T] float32, labels [N], per-clip silence stats)."""
        features, labels, stats = [], [], []
        for i, clip in enumerate(clips):
            feats, clip_stats = self(clip["waveform"], clip["sample_rate"])
            features.append(feats)
            labels.append(clip["label"])
            stats.append(clip_stats)
            if progress_every and (i + 1) % progress_every == 0:
                print(f"  featurized {i + 1}/{len(clips)} clips")
        x = torch.stack(features).unsqueeze(1)
        return x, torch.tensor(labels, dtype=torch.long), stats


# ---------------------------------------------------------------------------------------------------
# Training-time augmentation (Phase 2B Run B). Applied to training batches only -- never to
# validation or test, which the data sources enforce by only calling these when `training=True`.
# ---------------------------------------------------------------------------------------------------
def simulate_low_bitrate(waves, generator, downsample_factor=4, quant_bits=6, mu=255.0, p_apply=0.3):
    """Cheap proxy for a low-bitrate lossy-codec (e.g. MP3) round-trip: linear-interpolation
    downsample/upsample (the bandwidth loss a low bitrate causes) plus mu-law companding quantized to
    `quant_bits` (the quantization noise a low bitrate causes). This is NOT a real MP3 encode/decode --
    no differentiable, per-batch-fast MP3 codec exists -- it is a documented approximation of the two
    dominant artifacts a low-bitrate codec introduces, applied to a random subset of clips per batch.
    waves: [batch, samples] on any device, expected in [-1, 1]."""
    batch, n = waves.shape
    device = waves.device
    apply = (torch.rand(batch, 1, generator=generator, device=device) < p_apply).float()

    narrow = F.interpolate(waves.unsqueeze(1), size=max(n // downsample_factor, 1), mode="linear",
                           align_corners=False)
    wide = F.interpolate(narrow, size=n, mode="linear", align_corners=False).squeeze(1)

    x = wide.clamp(-1.0, 1.0)
    companded = torch.sign(x) * torch.log1p(mu * x.abs()) / math.log1p(mu)
    levels = 2 ** quant_bits - 1
    quantized = torch.round((companded + 1) / 2 * levels) / levels * 2 - 1
    expanded = torch.sign(quantized) * (torch.expm1(quantized.abs() * math.log1p(mu)) / mu)

    return waves + apply * (expanded - waves)


def augment_waveforms(waves, generator, snr_db=(10.0, 40.0), gain_db=(-6.0, 6.0), p_noise=0.5,
                      codec_sim=False):
    """Additive white noise at a random per-clip SNR (applied to a random half of clips) and a random
    per-clip gain. waves: [batch, samples] on any device. `codec_sim=True` (Phase 2C Run F only) adds
    `simulate_low_bitrate` on top -- Phase 2B explicitly skipped a codec augmentation for time."""
    batch = waves.shape[0]
    device = waves.device

    def uniform(lo, hi):
        return lo + (hi - lo) * torch.rand(batch, 1, generator=generator, device=device)

    signal_power = waves.pow(2).mean(dim=1, keepdim=True).clamp_min(1e-10)
    noise_power = signal_power / 10 ** (uniform(*snr_db) / 10)
    noise = torch.randn(waves.shape, generator=generator, device=device) * noise_power.sqrt()
    apply = (torch.rand(batch, 1, generator=generator, device=device) < p_noise).float()
    waves = waves + apply * noise
    waves = waves * 10 ** (uniform(*gain_db) / 20)
    if codec_sim:
        waves = simulate_low_bitrate(waves, generator)
    return waves


def spec_augment(generator, n_freq_masks=2, max_freq_width=8, n_time_masks=2, max_time_width=40):
    """SpecAugment-style masking for the log-filterbank tensor [batch, filters, frames]. Masked
    regions are set to each clip's mean log energy, so masking doesn't inject an artificial floor."""

    def apply(log_fbank):
        batch, n_filters, n_frames = log_fbank.shape
        device = log_fbank.device
        out = log_fbank.clone()
        fill = log_fbank.mean(dim=(1, 2), keepdim=True).expand_as(out)
        filters = torch.arange(n_filters, device=device).view(1, -1, 1)
        frames = torch.arange(n_frames, device=device).view(1, 1, -1)
        mask = torch.zeros_like(out, dtype=torch.bool)
        for _ in range(n_freq_masks):
            width = torch.randint(0, max_freq_width + 1, (batch, 1, 1), generator=generator, device=device)
            start = (torch.rand(batch, 1, 1, generator=generator, device=device) * (n_filters - width).clamp_min(1)).long()
            mask |= (filters >= start) & (filters < start + width)
        for _ in range(n_time_masks):
            width = torch.randint(0, max_time_width + 1, (batch, 1, 1), generator=generator, device=device)
            start = (torch.rand(batch, 1, 1, generator=generator, device=device) * (n_frames - width).clamp_min(1)).long()
            mask |= (frames >= start) & (frames < start + width)
        return torch.where(mask, fill, out)

    return apply

In [ ]:
# ---- inlined from engines/voice/evaluate.py (edit the module, then rebuild) ----
"""EER, DET curve and the silence-leakage probe.

Convention throughout: score = "how bonafide is this", label 1 = bonafide (the target class).
  FAR (false accept rate) = spoof clips scored above threshold  -> accepted as bonafide
  FRR (false reject rate)  = bonafide clips scored below threshold -> rejected as spoof
EER is where those two curves cross, found by linear interpolation between the two bracketing
operating points rather than by taking the nearest point, which is only an approximation.
"""
import numpy as np
from sklearn.metrics import roc_curve


def compute_eer(scores, labels, positive_label=1):
    """Equal Error Rate. Returns eer, the threshold at it, and FAR/FRR there."""
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels)
    y = (labels == positive_label).astype(int)
    if y.sum() == 0 or y.sum() == len(y):
        return {"eer": float("nan"), "threshold": float("nan"), "far": float("nan"),
                "frr": float("nan"), "note": "only one class present; EER undefined"}

    fpr, tpr, thresholds = roc_curve(y, scores)
    far, frr = fpr, 1.0 - tpr  # FAR rises with threshold-looseness, FRR falls
    diff = far - frr
    sign_change = np.flatnonzero(np.sign(diff[:-1]) * np.sign(diff[1:]) < 0)
    if len(sign_change) == 0:  # curves touch rather than cross (ties/small samples)
        idx = int(np.nanargmin(np.abs(diff)))
        return {"eer": float((far[idx] + frr[idx]) / 2), "threshold": float(thresholds[idx]),
                "far": float(far[idx]), "frr": float(frr[idx]), "interpolated": False}
    i = int(sign_change[0])
    # Linear interpolation of the crossing between operating points i and i+1.
    weight = diff[i] / (diff[i] - diff[i + 1])
    eer = float(far[i] + weight * (far[i + 1] - far[i]))
    frr_at = float(frr[i] + weight * (frr[i + 1] - frr[i]))
    threshold = float(thresholds[i] + weight * (thresholds[i + 1] - thresholds[i]))
    return {"eer": (eer + frr_at) / 2, "threshold": threshold, "far": eer, "frr": frr_at,
            "interpolated": True}


def det_curve_points(scores, labels, positive_label=1, max_points=512):
    """FAR/FRR pairs for plotting a DET curve, thinned to at most `max_points`."""
    y = (np.asarray(labels) == positive_label).astype(int)
    fpr, tpr, thresholds = roc_curve(y, np.asarray(scores, dtype=np.float64))
    far, frr = fpr, 1.0 - tpr
    if len(far) > max_points:
        keep = np.unique(np.linspace(0, len(far) - 1, max_points).astype(int))
        far, frr, thresholds = far[keep], frr[keep], thresholds[keep]
    return {"far": far.tolist(), "frr": frr.tolist(), "thresholds": thresholds.tolist()}


def per_class_accuracy(scores, labels, threshold, positive_label=1):
    scores, labels = np.asarray(scores), np.asarray(labels)
    predicted_positive = scores > threshold
    is_positive = labels == positive_label
    return {
        "threshold": float(threshold),
        "bonafide_accuracy": float(predicted_positive[is_positive].mean()) if is_positive.any() else None,
        "spoof_accuracy": float((~predicted_positive[~is_positive]).mean()) if (~is_positive).any() else None,
        "overall_accuracy": float((predicted_positive == is_positive).mean()),
        "n_bonafide": int(is_positive.sum()),
        "n_spoof": int((~is_positive).sum()),
    }


def _stratified_resample(labels, rng, positive_label=1):
    """Resample with replacement *within* each class, so every resample keeps the original class
    counts -- a plain resample could drop the (small) bonafide class and leave EER undefined."""
    pos = np.flatnonzero(labels == positive_label)
    neg = np.flatnonzero(labels != positive_label)
    return np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])


def bootstrap_eer_ci(scores, labels, n_resamples=1000, seed=0, alpha=0.05, positive_label=1):
    """Percentile bootstrap CI for EER, stratified by class."""
    scores, labels = np.asarray(scores, dtype=np.float64), np.asarray(labels)
    rng = np.random.default_rng(seed)
    samples = []
    for _ in range(n_resamples):
        idx = _stratified_resample(labels, rng, positive_label)
        samples.append(compute_eer(scores[idx], labels[idx], positive_label)["eer"])
    samples = np.array(samples)
    return {
        "eer": compute_eer(scores, labels, positive_label)["eer"],
        "ci_low": float(np.quantile(samples, alpha / 2)),
        "ci_high": float(np.quantile(samples, 1 - alpha / 2)),
        "ci_width": float(np.quantile(samples, 1 - alpha / 2) - np.quantile(samples, alpha / 2)),
        "bootstrap_std": float(samples.std()),
        "n_resamples": n_resamples,
        "n_bonafide": int((labels == positive_label).sum()),
        "n_spoof": int((labels != positive_label).sum()),
    }


def bootstrap_ci_at_counts(scores, labels, n_bonafide, n_spoof, n_resamples=1000, seed=0, alpha=0.05,
                           positive_label=1):
    """CI width the same scores would have had with a smaller test set of the given class counts.
    Used to say how much the larger Phase 2B test set narrowed the CI versus Phase 2's, on the same
    model, instead of comparing CIs from different models."""
    scores, labels = np.asarray(scores, dtype=np.float64), np.asarray(labels)
    pos = np.flatnonzero(labels == positive_label)
    neg = np.flatnonzero(labels != positive_label)
    rng = np.random.default_rng(seed)
    samples = []
    for _ in range(n_resamples):
        idx = np.concatenate([rng.choice(pos, n_bonafide), rng.choice(neg, n_spoof)])
        samples.append(compute_eer(scores[idx], labels[idx], positive_label)["eer"])
    low, high = np.quantile(samples, [alpha / 2, 1 - alpha / 2])
    return {"n_bonafide": int(n_bonafide), "n_spoof": int(n_spoof), "ci_low": float(low),
            "ci_high": float(high), "ci_width": float(high - low)}


def bootstrap_eer_difference(scores_a, labels_a, scores_b, labels_b, n_resamples=1000, seed=0,
                             alpha=0.05, paired=False, positive_label=1):
    """Percentile bootstrap CI for EER(a) - EER(b).

    paired=True: a and b are two models scored on the *same* clips in the same order, so each resample
    draws one set of indices used for both -- this cancels the test-set sampling noise the two share.
    paired=False: two different test sets (e.g. a random split vs. a speaker-disjoint split), each
    resampled independently.
    """
    sa, la = np.asarray(scores_a, dtype=np.float64), np.asarray(labels_a)
    sb, lb = np.asarray(scores_b, dtype=np.float64), np.asarray(labels_b)
    if paired and (len(sa) != len(sb) or not np.array_equal(la, lb)):
        raise ValueError("paired bootstrap needs both models scored on the same clips in the same order")
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_resamples):
        ia = _stratified_resample(la, rng, positive_label)
        ib = ia if paired else _stratified_resample(lb, rng, positive_label)
        diffs.append(compute_eer(sa[ia], la[ia], positive_label)["eer"] -
                     compute_eer(sb[ib], lb[ib], positive_label)["eer"])
    low, high = np.quantile(diffs, [alpha / 2, 1 - alpha / 2])
    point = compute_eer(sa, la, positive_label)["eer"] - compute_eer(sb, lb, positive_label)["eer"]
    return {"difference": float(point), "ci_low": float(low), "ci_high": float(high),
            "excludes_zero": bool(low > 0 or high < 0), "paired": paired, "n_resamples": n_resamples}


def speaker_leakage_call(diff):
    """Phase 2 printed "speaker leakage" for any positive difference, including +0.0009. The rule is
    now: only a CI that excludes zero counts. `diff` = EER(speaker-disjoint) - EER(random split), so a
    positive, significant difference means the random split flatters the model."""
    ci = f"{diff['difference']:+.4f} EER, 95% CI [{diff['ci_low']:+.4f}, {diff['ci_high']:+.4f}]"
    if not diff["excludes_zero"]:
        return {"verdict": "not_detectable",
                "text": f"no leakage detectable at this sample size ({ci}; the CI includes zero)"}
    if diff["difference"] > 0:
        return {"verdict": "leakage_detected",
                "text": f"speaker leakage detected: the random split looks better ({ci}, CI excludes zero)"}
    return {"verdict": "random_split_worse",
            "text": f"the random split is significantly *worse* ({ci}) -- not leakage; investigate the splits"}


def per_attack_eer(scores, labels, attacks, positive_label=1):
    """EER of all bonafide clips against each spoofing attack separately."""
    scores, labels, attacks = np.asarray(scores), np.asarray(labels), np.asarray(attacks)
    bonafide = labels == positive_label
    out = {}
    for attack in sorted(set(attacks[~bonafide].tolist())):
        mask = bonafide | (attacks == attack)
        out[attack] = {"eer": compute_eer(scores[mask], labels[mask], positive_label)["eer"],
                       "n_spoof": int((attacks == attack).sum())}
    return out


def evaluate_scores(scores, labels, positive_label=1):
    eer = compute_eer(scores, labels, positive_label)
    return {
        **eer,
        "per_class_at_eer_threshold": per_class_accuracy(scores, labels, eer["threshold"], positive_label),
        "det_curve": det_curve_points(scores, labels, positive_label),
    }


def duration_leakage_probe(stats, labels, positive_label=1):
    """How much can clip timing alone tell you about the label?

    Scores each clip by a duration measure and computes the EER of that single feature. An EER far
    from 50% means the dataset's recording/segmentation artifacts leak the label, independent of any
    spoofing artifact -- the known ASVspoof silence-duration shortcut. Both directions are tried per
    measure, since which sign favours bonafide is itself an empirical question; the reported EER is
    the better (lower) of the two, i.e. the strongest cheat available from that measure.
    """
    labels = np.asarray(labels)
    out = {}
    for key in ("total_s", "lead_silence_s", "trail_silence_s", "speech_s", "silence_ratio"):
        values = np.array([s[key] for s in stats], dtype=np.float64)
        if np.allclose(values, values[0]):
            out[key] = {"eer": float("nan"), "note": "constant measure"}
            continue
        forward = compute_eer(values, labels, positive_label)["eer"]
        backward = compute_eer(-values, labels, positive_label)["eer"]
        better = min(forward, backward)
        y = (labels == positive_label).astype(float)
        out[key] = {
            "eer": float(better),
            "direction": "higher_is_bonafide" if forward <= backward else "lower_is_bonafide",
            "point_biserial_r": float(np.corrcoef(values, y)[0, 1]),
            "mean_bonafide": float(values[labels == positive_label].mean()),
            "mean_spoof": float(values[labels != positive_label].mean()),
        }
    return out


def leakage_verdict(probe, strong=0.35, mild=0.45):
    """Turn the probe into a stated finding. `strong`/`mild` are EER thresholds: the further below
    50% a single duration measure gets, the more label information timing alone carries."""
    usable = {k: v["eer"] for k, v in probe.items() if not np.isnan(v["eer"])}
    if not usable:
        return {"level": "unknown", "worst_measure": None, "worst_eer": None,
                "text": "No usable duration measure; leakage could not be assessed."}
    worst_measure = min(usable, key=usable.get)
    worst = usable[worst_measure]
    level = "strong" if worst < strong else "mild" if worst < mild else "none"
    text = {
        "strong": f"Timing alone separates the classes well ({worst_measure} EER {worst:.4f}), so this data "
                  f"carries the known ASVspoof silence-duration shortcut.",
        "mild": f"Timing alone is weakly predictive ({worst_measure} EER {worst:.4f}) -- some leakage, but far "
                f"from a free solution.",
        "none": f"No meaningful timing leakage: the best duration measure only reaches EER {worst:.4f} "
                f"({worst_measure}), close to the 0.5 of a coin flip.",
    }[level]
    return {"level": level, "worst_measure": worst_measure, "worst_eer": float(worst), "text": text}

In [ ]:
# ---- inlined from engines/voice/model.py (edit the module, then rebuild) ----
"""LCNN (Light CNN) spoof detector, the SSL layer-weighted head, and a shared training loop.

LCNN is the CNN architecture used by the ASVspoof baselines. Its defining piece is the
Max-Feature-Map (MFM) activation: a conv produces 2k channels, which are split in half and combined
with an element-wise max, halving the channels. That acts as a learned feature selector and keeps
the network small.

Every model outputs one logit per clip, where a higher score means "more bonafide" (label 1 =
bonafide), matching the EER convention in evaluate.py.
"""
import math

import numpy as np
import torch
import torch.nn.functional as F



class MFM(torch.nn.Module):
    """Max-Feature-Map over channels (dim 1)."""

    def forward(self, x):
        a, b = torch.chunk(x, 2, dim=1)
        return torch.maximum(a, b)


def conv_mfm(in_ch, out_ch, kernel_size, padding):
    return torch.nn.Sequential(
        torch.nn.Conv2d(in_ch, out_ch * 2, kernel_size=kernel_size, padding=padding),
        MFM(),
    )


class LCNN(torch.nn.Module):
    MIN_SIZE = 16  # four 2x max-pools: each input axis must be at least 2**4 long

    def __init__(self, n_features=60, n_frames=401, channels=(16, 24, 32, 32), dropout=0.3, fc_dim=80):
        super().__init__()
        if n_features < self.MIN_SIZE or n_frames < self.MIN_SIZE:
            raise ValueError(f"LCNN needs at least {self.MIN_SIZE} feature rows and frames, got "
                             f"{n_features}x{n_frames} -- use a longer input window")
        c1, c2, c3, c4 = channels
        self.blocks = torch.nn.Sequential(
            conv_mfm(1, c1, 5, 2),
            torch.nn.MaxPool2d(2),
            conv_mfm(c1, c1, 1, 0),
            torch.nn.BatchNorm2d(c1),
            conv_mfm(c1, c2, 3, 1),
            torch.nn.MaxPool2d(2),
            torch.nn.BatchNorm2d(c2),
            conv_mfm(c2, c2, 1, 0),
            torch.nn.BatchNorm2d(c2),
            conv_mfm(c2, c3, 3, 1),
            torch.nn.MaxPool2d(2),
            conv_mfm(c3, c3, 1, 0),
            torch.nn.BatchNorm2d(c3),
            conv_mfm(c3, c4, 3, 1),
            torch.nn.MaxPool2d(2),
        )
        with torch.no_grad():  # measure the flattened size instead of deriving it from the pooling arithmetic
            flat_dim = self.blocks(torch.zeros(1, 1, n_features, n_frames)).numel()
        self.head = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(flat_dim, fc_dim * 2),
            MFM(),  # operates on dim 1, which is the feature dim for a 2D tensor
            torch.nn.BatchNorm1d(fc_dim),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(fc_dim, 1),
        )

    def forward(self, x):
        return self.head(self.blocks(x)).squeeze(-1)


class SSLHead(torch.nn.Module):
    """Classifier over pooled self-supervised encoder features [batch, layers, dim].

    A learned softmax weight per encoder layer mixes the layers into one vector, then a small MLP
    scores it. The layer weights are reported after training: they say which depth of the encoder
    carried the spoofing information.
    """

    def __init__(self, n_layers, feat_dim, hidden=128, dropout=0.3):
        super().__init__()
        self.layer_logits = torch.nn.Parameter(torch.zeros(n_layers))
        self.mlp = torch.nn.Sequential(
            torch.nn.LayerNorm(feat_dim),
            torch.nn.Linear(feat_dim, hidden),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden, 1),
        )

    def forward(self, x):
        weights = torch.softmax(self.layer_logits, dim=0)
        return self.mlp((weights.view(1, -1, 1) * x).sum(dim=1)).squeeze(-1)

    def layer_weights(self):
        return torch.softmax(self.layer_logits.detach(), dim=0).cpu().tolist()

    def n_params(self):
        return sum(p.numel() for p in self.parameters())


class SSLSeqHead(torch.nn.Module):
    """Phase 2C Model E: a single-layer bidirectional LSTM over one frozen SSL layer's per-frame
    token sequence [batch, frames, hidden], mean-pooled over time, then a small MLP head.

    Only one SSL layer is used (picked from Model C's learned layer weights, logged in the notebook),
    so there is no per-layer weighting mechanism here -- unlike `SSLHead`, which mixes every layer.
    Kept deliberately small (BiLSTM hidden size 64 by default): the goal is to test whether *any*
    temporal modeling helps, not to add capacity as a confound. `n_added_params` lets the notebook log
    that confound directly against `SSLHead`'s parameter count.
    """

    def __init__(self, feat_dim, lstm_hidden=64, hidden=128, dropout=0.3):
        super().__init__()
        self.lstm = torch.nn.LSTM(feat_dim, lstm_hidden, num_layers=1, batch_first=True, bidirectional=True)
        out_dim = 2 * lstm_hidden
        self.mlp = torch.nn.Sequential(
            torch.nn.LayerNorm(out_dim),
            torch.nn.Linear(out_dim, hidden),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.mlp(out.mean(dim=1)).squeeze(-1)

    def n_params(self):
        return sum(p.numel() for p in self.parameters())


def batches(n, batch_size, shuffle, generator=None):
    order = torch.randperm(n, generator=generator) if shuffle else torch.arange(n)
    for start in range(0, n, batch_size):
        yield order[start:start + batch_size]


def score_model(model, fetch, indices, batch_size=64):
    """Scores (higher = more bonafide) for `indices`, in order. `fetch(idx, training=False)`."""
    model.eval()
    indices = np.asarray(indices)
    out = []
    with torch.no_grad():
        for start in range(0, len(indices), batch_size):
            out.append(model(fetch(indices[start:start + batch_size], False)).detach().float().cpu())
    return torch.cat(out).numpy().astype(np.float64) if out else np.zeros(0)


def train_model(model, fetch, labels, train_idx, pos_weight, device, val_idx=None, epochs=20,
                batch_size=32, lr=3e-4, weight_decay=1e-4, patience=5, seed=42, verbose=True):
    """Class-weighted BCE training with optional early stopping on validation EER.

    `fetch(idx, training)` returns the model input for clip indices `idx` on `device`; data sources
    apply augmentation only when `training` is True, so validation and test inputs are never
    augmented. `labels` is indexed by the same clip indices.
    """
    labels = torch.as_tensor(np.asarray(labels))
    train_idx = np.asarray(train_idx)
    generator = torch.Generator().manual_seed(seed)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    pos_weight_t = torch.tensor([pos_weight], device=device, dtype=torch.float32)

    history, best = [], {"eer": math.inf, "epoch": 0, "state": None}
    since_improve, val_unusable = 0, False
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, n_seen = 0.0, 0
        for positions in batches(len(train_idx), batch_size, shuffle=True, generator=generator):
            if len(positions) < 2:
                continue  # BatchNorm needs at least 2 samples
            idx = train_idx[positions.numpy()]
            xb = fetch(idx, True)
            yb = labels[idx].to(device).float()
            optimizer.zero_grad()
            loss = F.binary_cross_entropy_with_logits(model(xb), yb, pos_weight=pos_weight_t)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(idx)
            n_seen += len(idx)
        epoch_loss = total_loss / max(n_seen, 1)
        if not math.isfinite(epoch_loss):
            raise FloatingPointError(f"training diverged at epoch {epoch} (loss={epoch_loss})")

        entry = {"epoch": epoch, "train_loss": epoch_loss}
        if val_idx is not None and not val_unusable:
            entry["val_eer"] = compute_eer(score_model(model, fetch, val_idx), labels[np.asarray(val_idx)].numpy())["eer"]
            if not math.isfinite(entry["val_eer"]):
                # One-class validation set: EER is undefined, so model selection and early stopping
                # have nothing to act on. Say so loudly rather than silently keeping the last epoch.
                val_unusable = True
                print("  WARNING: validation EER is undefined (validation set has a single class) -- "
                      "disabling early stopping and best-epoch selection for this run")
            elif entry["val_eer"] < best["eer"] - 1e-6:
                best = {"eer": entry["val_eer"], "epoch": epoch,
                        "state": {k: v.detach().clone() for k, v in model.state_dict().items()}}
                since_improve = 0
            else:
                since_improve += 1
        history.append(entry)
        if verbose:
            print(f"  epoch {epoch:3d}  loss {epoch_loss:.4f}" +
                  (f"  val EER {entry['val_eer']:.4f}" if "val_eer" in entry else ""))
        if patience is not None and val_idx is not None and not val_unusable and since_improve >= patience:
            if verbose:
                print(f"  early stopping at epoch {epoch} (best val EER {best['eer']:.4f} at epoch {best['epoch']})")
            break

    if best["state"] is not None:
        model.load_state_dict(best["state"])
    usable_val = val_idx is not None and not val_unusable
    return {"history": history, "best_val_eer": best["eer"] if usable_val else None,
            "best_epoch": best["epoch"] if usable_val else len(history),
            "epochs_run": len(history), "validation_usable": usable_val}


def train_lcnn(model, x_train, y_train, pos_weight, device, epochs=20, batch_size=32, lr=3e-4,
               weight_decay=1e-4, x_val=None, y_val=None, patience=5, seed=42, verbose=True):
    """In-memory wrapper around `train_model` (the Phase 2 notebook's interface)."""
    n_train = len(x_train)
    x_all = x_train if x_val is None else torch.cat([x_train, x_val])
    y_all = y_train if y_val is None else torch.cat([y_train, y_val])
    val_idx = None if x_val is None else np.arange(n_train, n_train + len(x_val))
    return train_model(model, lambda idx, training: x_all[torch.as_tensor(idx)].to(device), y_all,
                       np.arange(n_train), pos_weight, device, val_idx=val_idx, epochs=epochs,
                       batch_size=batch_size, lr=lr, weight_decay=weight_decay, patience=patience,
                       seed=seed, verbose=verbose)


@torch.no_grad()
def predict_scores(model, x, device, batch_size=64):
    """Scores where higher = more bonafide, for an in-memory feature tensor."""
    return score_model(model, lambda idx, training: x[torch.as_tensor(idx)].to(device),
                       np.arange(len(x)), batch_size)

In [ ]:
# ---- inlined from engines/voice/features.py (edit the module, then rebuild) ----
"""On-disk feature cache for Phase 2B, and the data sources training reads from.

One decode pass over the selected clips writes everything later steps need, as memory-mapped
arrays, so a kernel crash mid-run never forces the audio to be downloaded and decoded again:

  wave_trimmed.npy     int16 [N, samples]   trimmed, fixed-length waveforms. Runs A and B both read
                                            this and compute LFCC on the GPU per batch -- B only adds
                                            augmentation, so A vs. B changes exactly one variable. The
                                            SSL encoder (Run C) reads it too.
  lfcc_<variant>.npy   float16 [N, F, T]    untrimmed and silence-ablation features (comparisons only).
  lfcc_silence_only.npy                     LFCC of a fixed window of genuine silence per clip.
  clip_stats.json                           silence stats + which clips have enough silence.
  manifest.json                             what was built; a matching manifest means "reuse".

Every waveform is quantized to int16 before its LFCC is computed, so cached and on-the-fly
features see the same signal precision.
"""
import json
import time
from pathlib import Path

import numpy as np
import torch


INT16_SCALE = 32767.0
LFCC_VARIANTS = ("untrimmed",) + SILENCE_ABLATIONS


def to_int16(wave):
    return np.clip(np.round(wave * INT16_SCALE), -32768, 32767).astype(np.int16)


def quantize(wave):
    return to_int16(wave).astype(np.float32) / INT16_SCALE


def _manifest_matches(path, expected):
    if not path.exists():
        return False
    try:
        return json.loads(path.read_text(encoding="utf-8")).get("config") == expected
    except (json.JSONDecodeError, OSError):
        return False


def build_feature_cache(clip_iter, n_clips, cache_dir, extractor, device, clip_samples, silence_samples,
                        top_db=30.0, fixed_silence_s=0.5, noise_dbfs=-60.0, seed=42, batch_size=128,
                        config_key=None, progress_every=2000):
    """Single pass: decode each clip once, build every waveform variant, LFCC them on `device` in
    batches, and write the memmaps. Reuses an existing cache when its manifest matches."""
    from numpy.lib.format import open_memmap

    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = cache_dir / "manifest.json"
    expected = {"n_clips": int(n_clips), "clip_samples": int(clip_samples), "silence_samples": int(silence_samples),
                "top_db": top_db, "fixed_silence_s": fixed_silence_s, "noise_dbfs": noise_dbfs,
                "variants": list(LFCC_VARIANTS), "key": config_key}
    if _manifest_matches(manifest_path, expected):
        print(f"reusing feature cache in {cache_dir}")
        return open_feature_cache(cache_dir)

    n_feat = extractor.n_features
    frames, silence_frames = extractor.n_frames(clip_samples), extractor.n_frames(silence_samples)
    waves = open_memmap(cache_dir / "wave_trimmed.npy", mode="w+", dtype=np.int16, shape=(n_clips, clip_samples))
    lfcc = {v: open_memmap(cache_dir / f"lfcc_{v}.npy", mode="w+", dtype=np.float16, shape=(n_clips, n_feat, frames))
            for v in LFCC_VARIANTS}
    lfcc_silence = open_memmap(cache_dir / "lfcc_silence_only.npy", mode="w+", dtype=np.float16,
                               shape=(n_clips, n_feat, silence_frames))
    stats_out, has_silence = [], np.zeros(n_clips, dtype=bool)
    pending = {v: [] for v in LFCC_VARIANTS + ("silence_only",)}

    def flush(variant, target):
        rows = pending[variant]
        if not rows:
            return
        batch = torch.from_numpy(np.stack([w for _, w in rows])).to(device)
        with torch.no_grad():
            feats = extractor(batch).half().cpu().numpy()
        for (row, _), f in zip(rows, feats):
            target[row] = f
        pending[variant] = []

    start_time = time.time()
    count = 0
    for i, clip in enumerate(clip_iter):
        wave, sr = resample_linear(np.asarray(clip["waveform"], dtype=np.float32), clip["sample_rate"])
        _, stats = trim_silence(wave, sr, top_db)
        stats_out.append({k: stats[k] for k in ("total_s", "lead_silence_s", "trail_silence_s",
                                                  "speech_s", "silence_ratio", "all_silence")})
        waves[i] = to_int16(fix_length(wave[stats["start_sample"]:stats["end_sample"]], clip_samples))

        rng = np.random.default_rng(seed + i)
        variant_waves = {"untrimmed": fix_length(wave, clip_samples)}
        for mode in SILENCE_ABLATIONS:
            variant_waves[mode] = fix_length(ablate_silence(wave, sr, mode, stats, fixed_silence_s, noise_dbfs, rng),
                                             clip_samples)
        for variant, w in variant_waves.items():
            pending[variant].append((i, quantize(w)))
            if len(pending[variant]) >= batch_size:
                flush(variant, lfcc[variant])

        window = silence_window(wave, stats, silence_samples)
        has_silence[i] = window is not None
        pending["silence_only"].append((i, quantize(window if window is not None else np.zeros(silence_samples, np.float32))))
        if len(pending["silence_only"]) >= batch_size:
            flush("silence_only", lfcc_silence)

        count = i + 1
        if progress_every and count % progress_every == 0:
            rate = count / max(time.time() - start_time, 1e-6)
            print(f"  featurized {count}/{n_clips} clips ({rate:.0f} clips/s)")
    if count != n_clips:
        raise RuntimeError(f"expected {n_clips} clips, got {count}")
    for variant in LFCC_VARIANTS:
        flush(variant, lfcc[variant])
    flush("silence_only", lfcc_silence)

    for array in [waves, lfcc_silence, *lfcc.values()]:
        array.flush()
    (cache_dir / "clip_stats.json").write_text(
        json.dumps({"stats": stats_out, "has_silence_window": has_silence.tolist()}), encoding="utf-8")
    manifest_path.write_text(json.dumps({"config": expected, "built_seconds": round(time.time() - start_time, 1)},
                                        indent=2), encoding="utf-8")
    return open_feature_cache(cache_dir)


def open_feature_cache(cache_dir):
    cache_dir = Path(cache_dir)
    extra = json.loads((cache_dir / "clip_stats.json").read_text(encoding="utf-8"))
    return {
        "wave_trimmed": np.load(cache_dir / "wave_trimmed.npy", mmap_mode="r"),
        "lfcc": {v: np.load(cache_dir / f"lfcc_{v}.npy", mmap_mode="r") for v in LFCC_VARIANTS},
        "lfcc_silence_only": np.load(cache_dir / "lfcc_silence_only.npy", mmap_mode="r"),
        "stats": extra["stats"],
        "has_silence_window": np.array(extra["has_silence_window"], dtype=bool),
    }


def cache_size_gb(cache_dir):
    return sum(p.stat().st_size for p in Path(cache_dir).glob("*") if p.is_file()) / 1e9


# ---------------------------------------------------------------------------------------------------
# Data sources: fetch(idx, training) -> model input on `device`. Augmentation only when training.
# ---------------------------------------------------------------------------------------------------
def lfcc_array_source(array, device):
    """Cached float16 LFCC [N, F, T] -> LCNN input [B, 1, F, T]."""
    def fetch(idx, training):
        return torch.from_numpy(np.asarray(array[np.asarray(idx)], dtype=np.float32)).unsqueeze(1).to(device)
    return fetch


def wave_lfcc_source(waves_int16, extractor, device, augment=False, seed=0):
    """int16 waveforms -> (optionally augmented) LFCC on `device` -> LCNN input [B, 1, F, T]."""
    generator = torch.Generator(device=device).manual_seed(seed) if augment else None

    def fetch(idx, training):
        w = torch.from_numpy(np.asarray(waves_int16[np.asarray(idx)], dtype=np.float32) / INT16_SCALE).to(device)
        apply = augment and training
        if apply:
            w = augment_waveforms(w, generator)
        with torch.no_grad():
            feats = extractor(w, filterbank_augment=spec_augment(generator) if apply else None)
        return feats.unsqueeze(1)
    return fetch


def array_source(array, device):
    """Cached pooled SSL features [N, layers, dim] -> SSLHead input. Also used as-is for Phase 2C
    Model E's per-frame sequence cache [N, frames, hidden] -> SSLSeqHead input, since both are just a
    memmap slice on the leading axis."""
    def fetch(idx, training):
        return torch.from_numpy(np.asarray(array[np.asarray(idx)], dtype=np.float32)).to(device)
    return fetch


def array_mean_only_source(pooled_array, hidden_size, device):
    """Phase 2C Model D: the same cached mean+std pooled SSL features as Model C, sliced to the mean
    half only. No extra encoder pass -- `ssl.py`'s `pooled()` already concatenates mean and std, so
    this ablates Model D's one variable (std pooling) by reading less of the existing cache."""
    def fetch(idx, training):
        arr = np.asarray(pooled_array[np.asarray(idx)], dtype=np.float32)[..., :hidden_size]
        return torch.from_numpy(arr).to(device)
    return fetch


def wave_ssl_sequence_source(waves_int16, ssl_encoder, layer, device):
    """int16 waveforms -> on-the-fly per-frame sequence of one frozen SSL layer (Phase 2C Model E).

    Deliberately not cached to disk: a full-dataset single-layer sequence cache is tens of GB (see
    `ssl.py`'s `build_ssl_sequence_cache`), and stacking it on top of the feature cache this notebook
    already builds overran Kaggle's available disk on a real run (Version 2, 2026-09-29 -- issue #25).
    Same trade as `wave_ssl_source`: no disk cost, more GPU time re-running the frozen encoder.
    """
    def fetch(idx, training):
        w = torch.from_numpy(np.asarray(waves_int16[np.asarray(idx)], dtype=np.float32) / INT16_SCALE).to(device)
        return ssl_encoder.sequence(w, layer)
    return fetch


def wave_ssl_source(waves_int16, ssl_encoder, device, augment=False, seed=0, codec_sim=False, pooled_cache=None):
    """int16 waveforms -> on-the-fly frozen-SSL mean+std pooled features (Phase 2C Model F).

    Augmentation must happen upstream of the frozen encoder's forward pass, so -- unlike every other
    SSL run, which reads the pre-computed cache in `ssl.py` -- training calls (`training=True`) re-run
    the (frozen, no-grad) encoder every batch. Slower than the cache, but the encoder's own weights
    never see a gradient.

    `pooled_cache`, if given (Phase 2B's already-built `ssl_features`, same encoder/pooling/clips): used
    for every non-training call instead of re-running the encoder. Validation/test are never augmented,
    so their features are identical to that cache's -- recomputing them from the waveform every epoch
    (as an earlier version of this function did) was pure wasted GPU time, since validation runs once
    per epoch for up to `SSL_EPOCHS` epochs and test runs once more after training.
    """
    generator = torch.Generator(device=device).manual_seed(seed) if augment else None

    def fetch(idx, training):
        idx = np.asarray(idx)
        if not training and pooled_cache is not None:
            return torch.from_numpy(np.asarray(pooled_cache[idx], dtype=np.float32)).to(device)
        w = torch.from_numpy(np.asarray(waves_int16[idx], dtype=np.float32) / INT16_SCALE).to(device)
        if augment and training:
            w = augment_waveforms(w, generator, codec_sim=codec_sim)
        return ssl_encoder.pooled(w)
    return fetch

In [ ]:
# ---- inlined from engines/voice/ssl.py (edit the module, then rebuild) ----
"""Frozen self-supervised speech encoder (wav2vec 2.0 / XLS-R family) as a front-end (Phase 2B Run C).

The encoder is frozen and run once over every clip. Each layer's frame sequence is pooled to its
mean and standard deviation and cached, so the classifier (`model.SSLHead`, a learned softmax weight
per layer + a small MLP) trains in seconds and extra seeds cost almost nothing. Fine-tuning the
encoder is deliberately not done in this phase.
"""
from pathlib import Path

import numpy as np
import torch



class SSLEncoder:
    def __init__(self, model_name=None, device=torch.device("cpu"), fp16=True, model=None):
        if model is None:
            from transformers import AutoModel

            model = AutoModel.from_pretrained(model_name)
        self.device = device
        self.half = bool(fp16) and device.type == "cuda"
        self.model = model.to(device).eval()
        if self.half:
            self.model.half()
        for param in self.model.parameters():
            param.requires_grad_(False)
        self.model_name = model_name or type(model).__name__
        self.n_layers = self.model.config.num_hidden_layers + 1  # + the CNN feature-projection output
        self.hidden_size = self.model.config.hidden_size

    @property
    def feature_dim(self):
        return 2 * self.hidden_size  # mean and std pooling

    def _normalize(self, waves):
        x = waves.to(self.device, torch.float32)
        return (x - x.mean(dim=1, keepdim=True)) / torch.sqrt(x.var(dim=1, keepdim=True, unbiased=False) + 1e-7)

    @torch.no_grad()
    def pooled(self, waves):
        """[batch, samples] float waveforms -> [batch, layers, 2*hidden] pooled features.

        Inputs are normalized to zero mean / unit variance per clip, which is what the wav2vec 2.0 /
        XLS-R feature extractors do (`do_normalize=True`).
        """
        x = self._normalize(waves)
        out = self.model(x.half() if self.half else x, output_hidden_states=True)
        hidden = torch.stack(out.hidden_states, dim=1).float()  # [batch, layers, frames, hidden]
        return torch.cat([hidden.mean(dim=2), hidden.std(dim=2)], dim=-1)

    def mean_only(self, pooled_features):
        """[*, 2*hidden] concatenated mean+std pooling -> [*, hidden] mean-only half (Phase 2C Model D:
        an ablation of `pooled`'s std half, not a new encoder pass -- see `build_ssl_cache`'s cache)."""
        return pooled_features[..., :self.hidden_size]

    @torch.no_grad()
    def sequence(self, waves, layer):
        """[batch, samples] -> [batch, frames, hidden] per-frame hidden states of one layer index
        (0 = the CNN feature-projection output, same indexing as `hidden_states`). Phase 2C Model E
        needs the full frame sequence, not the pooled summary `pooled` gives; caching every layer's
        full sequence is infeasible (see the notebook's cost note), so only one layer is kept."""
        x = self._normalize(waves)
        out = self.model(x.half() if self.half else x, output_hidden_states=True)
        return out.hidden_states[layer].float()


def build_ssl_cache(encoder, waves_int16, cache_path, batch_size=16, progress_every=2000):
    """Pooled features for every clip in `waves_int16` -> float16 memmap [N, layers, 2*hidden].
    Reuses an existing file of the right shape (e.g. after a kernel restart)."""
    from numpy.lib.format import open_memmap

    cache_path = Path(cache_path)
    shape = (len(waves_int16), encoder.n_layers, encoder.feature_dim)
    done_marker = cache_path.with_suffix(".done")
    if cache_path.exists() and done_marker.exists():
        existing = np.load(cache_path, mmap_mode="r")
        if existing.shape == shape:
            print(f"reusing SSL feature cache {cache_path}")
            return existing
    out = open_memmap(cache_path, mode="w+", dtype=np.float16, shape=shape)
    for start in range(0, len(waves_int16), batch_size):
        batch = torch.from_numpy(np.asarray(waves_int16[start:start + batch_size], dtype=np.float32) / INT16_SCALE)
        out[start:start + len(batch)] = encoder.pooled(batch).cpu().numpy().astype(np.float16)
        done = start + len(batch)
        if progress_every and (done // batch_size) % max(1, progress_every // batch_size) == 0:
            print(f"  SSL features {done}/{len(waves_int16)}")
    out.flush()
    done_marker.write_text("ok", encoding="utf-8")
    return np.load(cache_path, mmap_mode="r")


def build_ssl_sequence_cache(encoder, waves_int16, layer, cache_path, batch_size=16, progress_every=2000):
    """Per-frame hidden states of one layer for every clip -> float16 memmap [N, frames, hidden]
    (Phase 2C Model E). Frame count is fixed because every input clip is the same fixed sample count.
    Reuses an existing file of the right shape, same as `build_ssl_cache`."""
    from numpy.lib.format import open_memmap

    cache_path = Path(cache_path)
    done_marker = cache_path.with_suffix(".done")
    with torch.no_grad():
        probe = encoder.sequence(torch.from_numpy(np.asarray(waves_int16[:1], dtype=np.float32) / INT16_SCALE), layer)
    n_frames = probe.shape[1]
    shape = (len(waves_int16), n_frames, encoder.hidden_size)
    if cache_path.exists() and done_marker.exists():
        existing = np.load(cache_path, mmap_mode="r")
        if existing.shape == shape:
            print(f"reusing SSL sequence cache {cache_path}")
            return existing
    out = open_memmap(cache_path, mode="w+", dtype=np.float16, shape=shape)
    for start in range(0, len(waves_int16), batch_size):
        batch = torch.from_numpy(np.asarray(waves_int16[start:start + batch_size], dtype=np.float32) / INT16_SCALE)
        out[start:start + len(batch)] = encoder.sequence(batch, layer).cpu().numpy().astype(np.float16)
        done = start + len(batch)
        if progress_every and (done // batch_size) % max(1, progress_every // batch_size) == 0:
            print(f"  SSL sequence features {done}/{len(waves_int16)}")
    out.flush()
    done_marker.write_text("ok", encoding="utf-8")
    return np.load(cache_path, mmap_mode="r")


def tiny_wav2vec2_for_smoke():
    """A randomly initialised, tiny wav2vec 2.0 -- exercises the real transformers code path in the
    local smoke test without downloading a checkpoint. Never used for reported numbers."""
    from transformers import Wav2Vec2Config, Wav2Vec2Model

    config = Wav2Vec2Config(hidden_size=32, num_hidden_layers=2, num_attention_heads=2, intermediate_size=64,
                            conv_dim=(16,) * 7, num_conv_pos_embeddings=16, num_conv_pos_embedding_groups=4)
    torch.manual_seed(0)
    return Wav2Vec2Model(config)

In [ ]:
# ---- inlined from engines/voice/codec.py (edit the module, then rebuild) ----
"""Codec and channel conditions for Phase 2D (docs/PHASE_2D_CODEC_ROBUSTNESS.md + addendum 1).

Real codecs go through ffmpeg (decode -> encode -> decode, per clip, nothing stored); room echo and
low-SNR noise are synthetic and need no external data. Condition names are fixed here so a result
file, a test and the runbook always mean the same thing.

Training menu (used to build augmented training clips):
    aac32, aac32_via48k, mp3_64, opus24, echo, noise_lowsnr
Held out (evaluation only, never trained on):
    aac16, opus12, mulaw8k, mp3_32

Everything takes and returns 16 kHz mono float32. The caller trims and fixes the length afterwards, in
exactly the way the clean clip is prepared, so a codec's delay or padding is treated like the service
would treat it.
"""
import shutil
import subprocess
import tempfile
from pathlib import Path

import numpy as np

CODEC_SR = 16000

TRAINING_MENU = ("aac32", "aac32_via48k", "mp3_64", "opus24", "echo", "noise_lowsnr")
HELD_OUT = ("aac16", "opus12", "mulaw8k", "mp3_32")
ALL_CONDITIONS = TRAINING_MENU + HELD_OUT

# name -> (file extension, encoder arguments, sample rate to pass through before encoding or None)
CODEC_SPECS = {
    "aac32": (".m4a", ["-c:a", "aac", "-b:a", "32k"], None),
    "aac32_via48k": (".m4a", ["-c:a", "aac", "-b:a", "32k"], 48000),
    "mp3_64": (".mp3", ["-c:a", "libmp3lame", "-b:a", "64k"], None),
    "opus24": (".opus", ["-c:a", "libopus", "-b:a", "24k"], None),
    "aac16": (".m4a", ["-c:a", "aac", "-b:a", "16k"], None),
    "opus12": (".opus", ["-c:a", "libopus", "-b:a", "12k"], None),
    "mp3_32": (".mp3", ["-c:a", "libmp3lame", "-b:a", "32k"], None),
    # G.711 telephony: 8 kHz, 8-bit mu-law, then back to 16 kHz.
    "mulaw8k": (".wav", ["-ar", "8000", "-c:a", "pcm_mulaw"], None),
    # Used only when libopus is missing from the available ffmpeg; named so the substitution is visible.
    "aac24_for_opus24": (".m4a", ["-c:a", "aac", "-b:a", "24k"], None),
    "aac12_for_opus12": (".m4a", ["-c:a", "aac", "-b:a", "12k"], None),
}
SUBSTITUTES = {"opus24": "aac24_for_opus24", "opus12": "aac12_for_opus12"}
SYNTHETIC = ("echo", "noise_lowsnr")

ECHO_RT60_S = (0.2, 0.8)       # reverberation time of the synthetic room
NOISE_SNR_DB = (5.0, 20.0)     # additive white noise, signal-to-noise range
AUGMENTATIONS_PER_CLIP = 3


class FFmpegMissing(RuntimeError):
    pass


def find_ffmpeg():
    """Path to an ffmpeg binary: the system one if present, else the `imageio-ffmpeg` package's."""
    system = shutil.which("ffmpeg")
    if system:
        return system
    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception as exc:  # noqa: BLE001 -- the message below is what the owner needs to see
        raise FFmpegMissing(
            "No ffmpeg found: it is not on the PATH and the imageio-ffmpeg package could not provide one "
            f"({type(exc).__name__}). Install it with: pip install imageio-ffmpeg") from exc


def _run(ffmpeg, *args):
    done = subprocess.run([str(ffmpeg), "-y", "-loglevel", "error", *map(str, args)], capture_output=True, text=True)
    if done.returncode != 0:
        raise RuntimeError(f"ffmpeg failed: {done.stderr.strip()[:300]}")


def codec_roundtrip(wave, name, ffmpeg):
    """float32 16 kHz -> codec `name` -> float32 16 kHz."""
    import soundfile as sf

    ext, args, via_rate = CODEC_SPECS[name]
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        source = tmp / "in.wav"
        sf.write(source, np.asarray(wave, dtype=np.float32), CODEC_SR, subtype="PCM_16")
        current = source
        if via_rate:
            current = tmp / "up.wav"
            _run(ffmpeg, "-i", source, "-ar", via_rate, "-c:a", "pcm_s16le", current)
        coded = tmp / f"coded{ext}"
        _run(ffmpeg, "-i", current, *args, coded)
        back = tmp / "back.wav"
        _run(ffmpeg, "-i", coded, "-ac", 1, "-ar", CODEC_SR, "-c:a", "pcm_s16le", back)
        out, _ = sf.read(back, dtype="float32")
    return np.ascontiguousarray(out, dtype=np.float32)


def synthetic_rir(rng, rt60_s, sr=CODEC_SR):
    """Room impulse response: unit direct path plus Gaussian noise decaying 60 dB over `rt60_s`."""
    n = int(sr * rt60_s)
    t = np.arange(n) / sr
    tail = rng.normal(0.0, 1.0, n) * 10.0 ** (-3.0 * t / rt60_s)   # amplitude: -60 dB at t = rt60
    tail *= 0.3 / (np.sqrt(np.mean(tail ** 2)) + 1e-12)            # reverberant energy about -10 dB re direct
    tail[0] = 1.0
    return tail.astype(np.float32)


def apply_echo(wave, rng):
    rt60 = float(rng.uniform(*ECHO_RT60_S))
    rir = synthetic_rir(rng, rt60)
    n = len(wave) + len(rir) - 1
    size = 1 << (n - 1).bit_length()
    out = np.fft.irfft(np.fft.rfft(wave, size) * np.fft.rfft(rir, size), size)[:len(wave)]
    # keep the loudness of the input: the model should see reverberation, not a level change
    out *= np.sqrt(np.mean(wave.astype(np.float64) ** 2) + 1e-12) / (np.sqrt(np.mean(out ** 2)) + 1e-12)
    return np.clip(out, -1.0, 1.0).astype(np.float32)


def apply_noise(wave, rng):
    snr_db = float(rng.uniform(*NOISE_SNR_DB))
    power = float(np.mean(wave.astype(np.float64) ** 2)) + 1e-12
    noise = rng.normal(0.0, np.sqrt(power / 10.0 ** (snr_db / 10.0)), len(wave))
    return np.clip(wave + noise, -1.0, 1.0).astype(np.float32)


def apply_condition(wave, name, rng, ffmpeg=None):
    """Apply one named condition. `rng` only matters for the synthetic ones."""
    if name == "echo":
        return apply_echo(np.asarray(wave, dtype=np.float32), rng)
    if name == "noise_lowsnr":
        return apply_noise(np.asarray(wave, dtype=np.float32), rng)
    if name not in CODEC_SPECS:
        raise KeyError(f"unknown condition {name!r}")
    return codec_roundtrip(wave, name, ffmpeg or find_ffmpeg())


def condition_rng(seed, position, name):
    """Deterministic generator for one (clip, condition): the same clip always gets the same echo/noise."""
    return np.random.default_rng([int(seed), int(position), sum(ord(c) * (i + 1) for i, c in enumerate(name))])


def resolve_conditions(ffmpeg, probe=None):
    """Which named conditions this machine can really produce, with every substitution made explicit.

    Returns {"training": [...], "held_out": [...], "substituted": {asked: used}, "unavailable": {name: why}}.
    `probe(name)` should raise if the condition cannot be produced; the default runs a short real round trip.
    """
    def default_probe(name):
        codec_roundtrip(np.random.default_rng(0).normal(0, 0.1, CODEC_SR // 2).astype(np.float32), name, ffmpeg)

    probe = probe or default_probe
    usable, substituted, unavailable = {}, {}, {}
    for name in ALL_CONDITIONS:
        if name in SYNTHETIC:
            usable[name] = name
            continue
        try:
            probe(name)
            usable[name] = name
        except Exception as exc:  # noqa: BLE001
            unavailable[name] = f"{type(exc).__name__}: {str(exc)[:120]}"
            fallback = SUBSTITUTES.get(name)
            if fallback:
                try:
                    probe(fallback)
                    usable[name] = fallback
                    substituted[name] = fallback
                except Exception as exc2:  # noqa: BLE001
                    unavailable[fallback] = f"{type(exc2).__name__}: {str(exc2)[:120]}"
    return {"training": [usable[n] for n in TRAINING_MENU if n in usable],
            "held_out": [usable[n] for n in HELD_OUT if n in usable],
            "substituted": substituted, "unavailable": unavailable}


def assign_conditions(n_clips, menu, per_clip=AUGMENTATIONS_PER_CLIP, seed=42):
    """[n_clips, per_clip] indices into `menu`: for each clip, `per_clip` distinct conditions, fixed by seed."""
    if len(menu) < per_clip:
        raise ValueError(f"the training menu has {len(menu)} usable conditions; {per_clip} are needed per clip")
    out = np.zeros((n_clips, per_clip), dtype=np.int16)
    for i in range(n_clips):
        out[i] = np.random.default_rng([int(seed), i]).choice(len(menu), size=per_clip, replace=False)
    return out

In [ ]:
# ---- inlined from engines/voice/robust.py (edit the module, then rebuild) ----
"""Phase 2D machinery: resumable pooled-feature extraction for codec-processed clips, a disk budget check,
cost projection for the pilot, and the clip-clustered paired bootstrap used by the pre-registered criteria.

The encoder is frozen, so every version of a clip (clean, augmented, evaluation condition) is turned into
a pooled feature vector once and stored; the head trains on those vectors. Only the mean-pooled half is
kept (Model D uses nothing else): [25 layers, hidden] float16 = 51,200 bytes per version for XLS-R 300M.

Storage is a directory of chunk files, `chunk_00000.npz`, `chunk_00001.npz`, ... Each holds every version
produced for a fixed run of clips. A chunk is written to a temporary name and renamed when complete, so a
file that exists is a finished chunk; after a crash or a session limit the run resumes at the first
missing chunk. Nothing but pooled vectors, positions and condition indices is stored: no waveforms.
"""
import os
import shutil
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch


CHUNK_CLIPS = 256
DISK_FACTOR = 1.3
BYTES_PER_VALUE = 2  # float16


class DiskBudgetError(RuntimeError):
    pass


def prepare_clip(wave, sr, clip_samples, top_db=30.0):
    """Resample, trim leading/trailing silence, repeat-pad or crop to `clip_samples`, quantise to int16.
    The same steps (and the same trim rule) as the clean cache of Phase 2B, applied to a codec's output."""
    wave, sr = resample_linear(np.asarray(wave, dtype=np.float32), sr)
    _, stats = trim_silence(wave, sr, top_db)
    return to_int16(fix_length(wave[stats["start_sample"]:stats["end_sample"]], clip_samples))


# --- cost and disk ------------------------------------------------------------------------------------------
def version_counts(n_clips, n_trainval, n_eval_clips, n_eval_conditions, per_clip=3):
    clean = int(n_clips)
    aug = int(n_trainval) * per_clip
    evaluation = int(n_eval_clips) * int(n_eval_conditions)
    return {"clean": clean, "augmented": aug, "evaluation": evaluation, "total": clean + aug + evaluation}


def bytes_per_version(n_layers, hidden_size):
    return int(n_layers) * int(hidden_size) * BYTES_PER_VALUE


def check_disk(path, required_bytes, factor=DISK_FACTOR):
    """Plain-language stop if the free space under `path` is below `factor` x the cache size."""
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(path).free
    report = {"required_gb": required_bytes / 1e9, "needed_with_margin_gb": factor * required_bytes / 1e9,
              "free_gb": free / 1e9, "ok": free >= factor * required_bytes}
    if not report["ok"]:
        raise DiskBudgetError(
            f"Not enough free disk space. The feature cache needs about {report['required_gb']:.1f} GB "
            f"(and {factor:g}x that is required before starting: {report['needed_with_margin_gb']:.1f} GB), "
            f"but only {report['free_gb']:.1f} GB is free. Nothing was started. Lower EVAL_PER_CONDITION or "
            f"the number of training clips, or free space, and run again.")
    return report


def project_cost(measured_seconds, measured_versions, measured_bytes, total_versions):
    """Straight-line projection from the pilot. It is an estimate, not a promise: the pilot says so."""
    per_version = measured_seconds / max(measured_versions, 1)
    return {"seconds_per_version": per_version, "projected_seconds": per_version * total_versions,
            "projected_hours": per_version * total_versions / 3600.0,
            "projected_cache_gb": measured_bytes / max(measured_versions, 1) * total_versions / 1e9}


# --- chunk store --------------------------------------------------------------------------------------------
def chunk_name(index):
    return f"chunk_{int(index):05d}.npz"


def find_chunk(index, dirs):
    for d in dirs:
        candidate = Path(d) / chunk_name(index)
        if candidate.exists():
            return candidate
    return None


def load_chunks(dirs, kinds=("clean", "aug", "eval")):
    """Concatenate every finished chunk found in `dirs` (first directory wins per chunk index).
    Returns {kind: {"positions", "data", ...extra index arrays}}; missing kinds come back empty."""
    indices = sorted({int(p.stem.split("_")[1]) for d in dirs if Path(d).exists() for p in Path(d).glob("chunk_*.npz")
                      if not p.stem.endswith(".tmp")})
    parts = {k: [] for k in kinds}
    for index in indices:
        with np.load(find_chunk(index, dirs)) as z:
            for kind in kinds:
                if f"{kind}_positions" in z.files and len(z[f"{kind}_positions"]):
                    parts[kind].append({name[len(kind) + 1:]: z[name] for name in z.files if name.startswith(kind + "_")}
                                       | {"data": z[kind]})
    out = {}
    for kind, items in parts.items():
        if not items:
            out[kind] = {"positions": np.zeros(0, dtype=np.int64), "data": None}
            continue
        merged = {key: np.concatenate([it[key] for it in items]) for key in items[0]}
        out[kind] = merged
    return out


def extract_features(get_clip, positions, encoder, clip_samples, out_dir, trainval, aug_conditions, aug_assign,
                     eval_positions, eval_conditions, ffmpeg=None, resume_dirs=(), seed=42, chunk_clips=CHUNK_CLIPS,
                     batch_size=16, workers=4, progress_every=1, deadline=None, log=print):
    """Produce every version of every clip in `positions`, chunk by chunk, skipping finished chunks.

    get_clip(position) -> (waveform float32, sample_rate).     encoder: SSLEncoder (frozen).
    trainval:       set of positions that get the clean version plus augmented versions.
    aug_assign:     {position: [index into aug_conditions] * 3}.
    eval_positions: set of positions that get one version per name in eval_conditions.
    deadline:       time.time() value after which the run stops cleanly at the next chunk boundary.
    Returns a stats dict (versions made, seconds split into decode / conditions / encoder, stopped_early).
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    dirs = [out_dir, *map(Path, resume_dirs)]
    hidden = encoder.hidden_size
    positions = list(positions)
    n_chunks = (len(positions) + chunk_clips - 1) // chunk_clips
    stats = {"versions": 0, "chunks_done": 0, "chunks_reused": 0, "seconds": 0.0, "seconds_decode": 0.0,
             "seconds_conditions": 0.0, "seconds_encoder": 0.0, "stopped_early": False, "n_chunks": n_chunks}
    started = time.time()
    pool = ThreadPoolExecutor(max_workers=max(1, workers))
    try:
        for c in range(n_chunks):
            if find_chunk(c, dirs) is not None:
                stats["chunks_reused"] += 1
                continue
            if deadline is not None and time.time() > deadline:
                stats["stopped_early"] = True
                log(f"  time limit reached after chunk {c - 1}; stopping cleanly. Run again to continue.")
                break
            chunk_positions = positions[c * chunk_clips:(c + 1) * chunk_clips]

            t = time.time()
            waves = {p: get_clip(p) for p in chunk_positions}
            stats["seconds_decode"] += time.time() - t

            jobs = []   # (kind, position, condition name or None, extra index)
            for p in chunk_positions:
                jobs.append(("clean", p, None, 0))
                if p in trainval:
                    for slot, ci in enumerate(aug_assign[p]):
                        jobs.append(("aug", p, aug_conditions[int(ci)], int(slot) * 1000 + int(ci)))
                if p in eval_positions:
                    for ei, name in enumerate(eval_conditions):
                        jobs.append(("eval", p, name, ei))

            def make(job):
                kind, p, name, _ = job
                wave, sr = waves[p]
                if name is None:
                    return prepare_clip(wave, sr, clip_samples)
                wave16, _ = resample_linear(np.asarray(wave, dtype=np.float32), sr)
                rng = condition_rng(seed + (1_000_000 if kind == "eval" else 0), p, name)
                return prepare_clip(apply_condition(wave16, name, rng, ffmpeg), 16000, clip_samples)

            t = time.time()
            clips = list(pool.map(make, jobs))
            stats["seconds_conditions"] += time.time() - t

            t = time.time()
            feats = np.zeros((len(jobs), encoder.n_layers, hidden), dtype=np.float16)
            for s in range(0, len(jobs), batch_size):
                batch = torch.from_numpy(np.stack(clips[s:s + batch_size]).astype(np.float32) / INT16_SCALE)
                pooled = encoder.pooled(batch)[..., :hidden]
                feats[s:s + len(batch)] = pooled.cpu().numpy().astype(np.float16)
            stats["seconds_encoder"] += time.time() - t

            arrays = {}
            for kind in ("clean", "aug", "eval"):
                idx = [i for i, j in enumerate(jobs) if j[0] == kind]
                arrays[f"{kind}_positions"] = np.array([jobs[i][1] for i in idx], dtype=np.int64)
                arrays[kind] = feats[idx] if idx else np.zeros((0, encoder.n_layers, hidden), dtype=np.float16)
                if kind == "aug":
                    arrays["aug_slot"] = np.array([jobs[i][3] // 1000 for i in idx], dtype=np.int16)
                    arrays["aug_cond"] = np.array([jobs[i][3] % 1000 for i in idx], dtype=np.int16)
                if kind == "eval":
                    arrays["eval_cond"] = np.array([jobs[i][3] for i in idx], dtype=np.int16)
            tmp = out_dir / f"chunk_{c:05d}.tmp.npz"
            np.savez(tmp, **arrays)
            os.replace(tmp, out_dir / chunk_name(c))

            stats["versions"] += len(jobs)
            stats["chunks_done"] += 1
            stats["seconds"] = time.time() - started
            if progress_every and (stats["chunks_done"] % progress_every == 0 or c == n_chunks - 1):
                rate = stats["seconds"] / max(stats["versions"], 1)
                log(f"  chunk {c + 1}/{n_chunks}: {stats['versions']} versions made, {rate:.2f} s per version "
                    f"(decode {stats['seconds_decode']:.0f}s, conditions {stats['seconds_conditions']:.0f}s, "
                    f"encoder {stats['seconds_encoder']:.0f}s)")
    finally:
        pool.shutdown(wait=True)
    stats["seconds"] = time.time() - started
    return stats


# --- evaluation helpers -------------------------------------------------------------------------------------
def clustered_paired_bootstrap(scores_a, scores_b, labels, groups, n_resamples=1000, seed=0, alpha=0.05):
    """Percentile bootstrap of EER(a) - EER(b) that resamples whole clips.

    Rows are (clip, condition) versions; several rows share a clip, so resampling rows would treat copies of
    one clip as independent and make the interval too narrow. Clips are resampled within class (bonafide /
    spoof) and all of a drawn clip's rows are kept, for both models alike.
    """
    sa, sb = np.asarray(scores_a, dtype=np.float64), np.asarray(scores_b, dtype=np.float64)
    labels, groups = np.asarray(labels), np.asarray(groups)
    unique, first = np.unique(groups, return_index=True)
    group_label = labels[first]
    rows_of = {g: np.flatnonzero(groups == g) for g in unique}
    pos, neg = unique[group_label == 1], unique[group_label != 1]
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_resamples):
        drawn = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        idx = np.concatenate([rows_of[g] for g in drawn])
        diffs.append(compute_eer(sa[idx], labels[idx])["eer"] - compute_eer(sb[idx], labels[idx])["eer"])
    low, high = np.quantile(diffs, [alpha / 2, 1 - alpha / 2])
    point = compute_eer(sa, labels)["eer"] - compute_eer(sb, labels)["eer"]
    return {"difference": float(point), "ci_low": float(low), "ci_high": float(high),
            "excludes_zero": bool(low > 0 or high < 0), "clustered": True, "n_resamples": n_resamples,
            "n_clips_bonafide": int(len(pos)), "n_clips_spoof": int(len(neg)), "n_rows": int(len(labels))}


def criterion_status(difference):
    """Reads a bootstrap difference dict (F2 minus control) the way the pre-registered criteria do.
    Negative difference = F2 has the lower (better) EER.
    not_worse: the interval does not lie wholly above zero (criteria 1 and 2a).
    better:    the interval lies wholly below zero (criterion 2b)."""
    return {"not_worse": bool(difference["ci_low"] <= 0), "better": bool(difference["ci_high"] < 0)}


def share_flagged(scores, labels, threshold):
    """Share of genuine (label 1) clips scored below `threshold`, i.e. what a caller would see flagged."""
    scores, labels = np.asarray(scores, dtype=np.float64), np.asarray(labels)
    genuine = scores[labels == 1]
    return float((genuine < threshold).mean()) if len(genuine) else float("nan")

## Step 1 -- the same data and the same split as Model D

In [ ]:
t0 = time.time()
if SMOKE_TEST:
    synthetic = make_synthetic_asvspoof(240, n_attacks=6, bonafide_pad_s=(0.12, 0.3))
    table = None
    meta = {"label": np.array([c["label"] for c in synthetic]),
            "utterance_id": [c["utterance_id"] for c in synthetic],
            "speaker_id": [c["speaker_id"] for c in synthetic]}
    dataset_name = "synthetic (smoke test)"
else:
    print(f"downloading {ASVSPOOF_DATASET} (4.39 GB parquet, cached by `datasets`) ...")
    table = load_asvspoof_table()
    meta = table_metadata(table)
    dataset_name = f"{ASVSPOOF_DATASET} (split '{ASVSPOOF_SPLIT}')"
print(f"metadata for {len(meta['label'])} clips in {time.time() - t0:.0f}s")

selected = select_subset(meta["label"], SPOOF_PER_BONAFIDE, seed=SEED)
labels_np = meta["label"][selected].astype(np.int64)
speakers = [meta["speaker_id"][i] for i in selected]
N = len(selected)
print(f"{N} clips selected ({int((labels_np == BONAFIDE).sum())} bonafide / {int((labels_np == SPOOF).sum())} spoof)")


def get_clip(position):
    """Decode clip number `position` of the selected subset: (float32 waveform, sample rate)."""
    if SMOKE_TEST:
        clip = synthetic[int(selected[position])]
        return clip["waveform"], clip["sample_rate"]
    return _decode_audio(table[int(selected[position])]["audio"])


meta_clips = [{"speaker_id": speakers[i], "label": int(labels_np[i])} for i in range(N)]


def sub_split(indices, fraction, seed):
    subset = [meta_clips[i] for i in indices]
    left, right = speaker_disjoint_split(subset, fraction, seed)
    return [indices[i] for i in left], [indices[i] for i in right]


def counts_of(idx):
    lab = labels_np[np.asarray(idx, dtype=np.int64)]
    return {"bonafide": int((lab == BONAFIDE).sum()), "spoof": int((lab == SPOOF).sum())}


# Identical calls to notebook 04's P1 split.
trainval, test = speaker_disjoint_split(meta_clips, TEST_FRACTION, SEED)
train, val = sub_split(trainval, VAL_FRACTION, SEED + 1)
test_idx_hash = __import__("hashlib").sha256(np.asarray(test, dtype=np.int64).tobytes()).hexdigest()
split_identity = {
    "counts": {"train": counts_of(train), "val": counts_of(val), "test": counts_of(test)},
    "shared_speakers_train_test": len({speakers[i] for i in train} & {speakers[i] for i in test}),
    "test_idx_sha256": test_idx_hash, "expected_test_idx_sha256": EXPECTED_TEST_IDX_SHA256,
    "matches_model_d_hash": test_idx_hash == EXPECTED_TEST_IDX_SHA256,
    "matches_model_d_counts": {k: counts_of(v) for k, v in (("train", train), ("val", val), ("test", test))} == EXPECTED_P1,
}
print(json.dumps(split_identity, indent=1))
if not SMOKE_TEST:
    # Stop rather than compare against a different split: every number below would be unusable.
    assert split_identity["matches_model_d_counts"], "P1 split counts differ from Model D's recorded split"
    assert split_identity["matches_model_d_hash"], "P1 test indices differ from Model D's recorded split"
assert split_identity["shared_speakers_train_test"] == 0, "speaker overlap between train and test"
save_progress("split_identity", split_identity)

In [ ]:
# Evaluation subset: all bonafide test clips plus spoofs up to EVAL_PER_CONDITION clips (seeded, fixed once).
test_arr = np.asarray(test, dtype=np.int64)
if EVAL_PER_CONDITION is None or len(test_arr) <= EVAL_PER_CONDITION:
    eval_positions = test_arr
else:
    bonafide_test = test_arr[labels_np[test_arr] == BONAFIDE]
    spoof_test = test_arr[labels_np[test_arr] == SPOOF]
    n_spoof = max(1, EVAL_PER_CONDITION - len(bonafide_test))
    rng = np.random.default_rng(SEED + 7)
    eval_positions = np.sort(np.concatenate([bonafide_test, rng.choice(spoof_test, min(n_spoof, len(spoof_test)), replace=False)]))
eval_set = set(int(p) for p in eval_positions)
trainval_set = set(int(p) for p in trainval)

# What this machine can really produce. Substitutions and missing codecs are recorded, not hidden.
FFMPEG = find_ffmpeg()
CONDITIONS = resolve_conditions(FFMPEG)
print("ffmpeg:", FFMPEG)
print("training conditions:", CONDITIONS["training"])
print("held-out conditions:", CONDITIONS["held_out"])
if CONDITIONS["substituted"] or CONDITIONS["unavailable"]:
    print("SUBSTITUTED:", CONDITIONS["substituted"], "| UNAVAILABLE:", CONDITIONS["unavailable"])
if len(CONDITIONS["training"]) < AUGMENTATIONS_PER_CLIP:
    raise RuntimeError(f"only {len(CONDITIONS['training'])} training conditions can be produced here")
if not CONDITIONS["held_out"]:
    raise RuntimeError("no held-out condition can be produced here, so unseen codecs cannot be tested")
EVAL_CONDITIONS = CONDITIONS["training"] + CONDITIONS["held_out"]
AUG_ASSIGN_ARRAY = assign_conditions(N, CONDITIONS["training"], AUGMENTATIONS_PER_CLIP, seed=SEED)
AUG_ASSIGN = {int(p): AUG_ASSIGN_ARRAY[int(p)].tolist() for p in trainval}

In [ ]:
# Encoder: XLS-R 300M, frozen, exactly as Model D's. No silent fallback to a different encoder: a different
# encoder would make "same as Model D" false.
if SMOKE_TEST:
    ssl_encoder = SSLEncoder(model=tiny_wav2vec2_for_smoke(), device=DEVICE, fp16=False)
    ssl_name = "tiny random wav2vec2 (smoke test)"
else:
    ssl_encoder = SSLEncoder(SSL_MODEL, device=DEVICE, fp16=True)
    ssl_name = SSL_MODEL
HIDDEN, N_LAYERS = ssl_encoder.hidden_size, ssl_encoder.n_layers
VERSION_BYTES = bytes_per_version(N_LAYERS, HIDDEN)
print(f"encoder {ssl_name}: {N_LAYERS} layers x {HIDDEN} values, {VERSION_BYTES / 1000:.1f} KB per stored version")

PLAN = version_counts(N, len(trainval), len(eval_positions), len(EVAL_CONDITIONS), AUGMENTATIONS_PER_CLIP)
PLAN["bytes_per_version"] = VERSION_BYTES
PLAN["total_cache_gb"] = PLAN["total"] * VERSION_BYTES / 1e9
PLAN["eval_clips_per_condition"] = {"total": int(len(eval_positions)), **counts_of(eval_positions)}
print("planned versions:", json.dumps(PLAN, indent=1))
save_progress("plan", {**PLAN, "conditions": CONDITIONS, "ssl_model": ssl_name, "eval_per_condition_cap": EVAL_PER_CONDITION,
                       "augmentations_per_clip": AUGMENTATIONS_PER_CLIP})

## Step 2 -- pilot: speed and size on a small sample, then stop

In [ ]:
if RUN_MODE == "PILOT":
    rng = np.random.default_rng(SEED + 99)
    n_tv = max(1, int(round(PILOT_CLIPS * 0.8)))
    pilot_tv = rng.choice(np.asarray(trainval, dtype=np.int64), min(n_tv, len(trainval)), replace=False)
    pilot_te = rng.choice(eval_positions, min(PILOT_CLIPS - len(pilot_tv), len(eval_positions)), replace=False)
    pilot_positions = sorted(int(p) for p in np.concatenate([pilot_tv, pilot_te]))
    pilot_set = set(pilot_positions)
    shutil.rmtree(PILOT_DIR, ignore_errors=True)
    print(f"PILOT: {len(pilot_positions)} clips ({len(pilot_tv)} training, {len(pilot_te)} test)")
    stats = extract_features(get_clip, pilot_positions, ssl_encoder, CLIP_SAMPLES, PILOT_DIR,
                             trainval=trainval_set & pilot_set, aug_conditions=CONDITIONS["training"],
                             aug_assign=AUG_ASSIGN, eval_positions=eval_set & pilot_set,
                             eval_conditions=EVAL_CONDITIONS, ffmpeg=FFMPEG, seed=SEED, chunk_clips=CHUNK_SIZE,
                             batch_size=SSL_BATCH, workers=WORKERS)
    measured_bytes = sum(p.stat().st_size for p in PILOT_DIR.glob("chunk_*.npz"))
    projection = project_cost(stats["seconds"], stats["versions"], measured_bytes, PLAN["total"])
    disk = shutil.disk_usage(CACHE_DIR if CACHE_DIR.exists() else OUTPUT_DIR)
    pilot = {"clips": len(pilot_positions), "versions_made": stats["versions"],
             "seconds_total": round(stats["seconds"], 1), "seconds_per_clip": round(stats["seconds"] / len(pilot_positions), 2),
             "seconds_per_version": round(projection["seconds_per_version"], 3),
             "seconds_split": {k: round(stats[k], 1) for k in ("seconds_decode", "seconds_conditions", "seconds_encoder")},
             "projected_versions_full": PLAN["total"], "projected_extraction_hours": round(projection["projected_hours"], 2),
             "projected_cache_gb": round(projection["projected_cache_gb"], 2), "free_disk_gb": round(disk.free / 1e9, 1),
             "disk_ok_with_margin": disk.free >= DISK_FACTOR * projection["projected_cache_gb"] * 1e9,
             "note": "straight-line projection from this sample; head training time is not included (minutes)."}
    save_progress("pilot", pilot)
    print(json.dumps(pilot, indent=1))
    print(f"\nPILOT DONE. Full extraction is projected at about {pilot['projected_extraction_hours']:.1f} hours and "
          f"{pilot['projected_cache_gb']:.1f} GB of cache. Free disk: {pilot['free_disk_gb']:.0f} GB.")
    print("If that fits, set RUN_MODE = 'FULL' in the setup cell and run again (Save & Run All).")
    shutil.rmtree(PILOT_DIR, ignore_errors=True)

## Step 3 -- full extraction (chunked, resumable)

In [ ]:
FULL = RUN_MODE == "FULL"
if FULL:
    required = PLAN["total"] * VERSION_BYTES
    disk_report = check_disk(CACHE_DIR, required)
    print(f"cache needs about {disk_report['required_gb']:.1f} GB; free {disk_report['free_gb']:.1f} GB (margin x{DISK_FACTOR:g}) -- ok")
    deadline = RUN_START + MAX_EXTRACT_HOURS * 3600
    extraction = extract_features(get_clip, list(range(N)), ssl_encoder, CLIP_SAMPLES, CACHE_DIR,
                                  trainval=trainval_set, aug_conditions=CONDITIONS["training"], aug_assign=AUG_ASSIGN,
                                  eval_positions=eval_set, eval_conditions=EVAL_CONDITIONS, ffmpeg=FFMPEG,
                                  resume_dirs=RESUME_DIRS, seed=SEED, chunk_clips=CHUNK_SIZE, batch_size=SSL_BATCH,
                                  workers=WORKERS, deadline=deadline)
    save_progress("extraction", {k: (round(v, 1) if isinstance(v, float) else v) for k, v in extraction.items()})
    print(json.dumps(RESULTS["extraction"], indent=1))
    if extraction["stopped_early"]:
        print("\nSTOPPED EARLY on the time limit. Nothing is lost: the finished chunks are in the output. Start a new "
              "session with this output attached as input, set TRUSTMESH_RESUME_DIRS to its cache folder, and run again.")
        FULL = False

## Step 4 -- train the control (clean only) and F2 (clean + augmented), 3 seeds

In [ ]:
head_results, SEED_SCORES = {}, {}
if FULL:
    loaded = load_chunks([CACHE_DIR, *RESUME_DIRS], kinds=("clean", "aug"))
    clean, aug = loaded["clean"], loaded["aug"]
    assert len(clean["positions"]) == N, f"expected {N} clean versions, found {len(clean['positions'])}"
    pool = np.concatenate([clean["data"], aug["data"]])
    pool_positions = np.concatenate([clean["positions"], aug["positions"]])
    pool_is_aug = np.concatenate([np.zeros(len(clean["positions"]), bool), np.ones(len(aug["positions"]), bool)])
    del loaded, clean, aug
    pool_labels = labels_np[pool_positions]
    train_set, val_set = set(int(p) for p in train), set(int(p) for p in val)
    in_train = np.fromiter((int(p) in train_set for p in pool_positions), bool, len(pool_positions))
    in_val = np.fromiter((int(p) in val_set for p in pool_positions), bool, len(pool_positions))
    control_rows = np.flatnonzero(in_train & ~pool_is_aug)
    f2_rows = np.flatnonzero(in_train)
    val_rows = np.flatnonzero(in_val)                    # clean + augmented validation clips, both models
    val_clean_rows = np.flatnonzero(in_val & ~pool_is_aug)
    print(f"control trains on {len(control_rows)} rows, F2 on {len(f2_rows)}, validation {len(val_rows)} "
          f"({len(val_clean_rows)} clean + {len(val_rows) - len(val_clean_rows)} augmented)")

    def pool_fetch(idx, training):
        return torch.from_numpy(np.asarray(pool[np.asarray(idx)], dtype=np.float32)).to(DEVICE)

    def train_head(name, rows, seed):
        torch.manual_seed(seed)
        np.random.seed(seed)
        model = SSLHead(N_LAYERS, HIDDEN)
        lab = pool_labels[rows]
        weight = float((lab == SPOOF).sum() / max((lab == BONAFIDE).sum(), 1))
        t_start = time.time()
        fit = train_model(model, pool_fetch, pool_labels, rows, weight, DEVICE, val_idx=val_rows, epochs=SSL_EPOCHS,
                          batch_size=BATCH_SIZE, lr=SSL_LR, patience=SSL_PATIENCE, seed=seed, verbose=False)
        head_results[name] = {"seed": seed, "train_rows": int(len(rows)), "n_params": model.n_params(),
                              "best_val_eer_mixed": fit["best_val_eer"], "best_epoch": fit["best_epoch"],
                              "epochs_run": fit["epochs_run"], "train_seconds": round(time.time() - t_start, 1),
                              "layer_weights": model.layer_weights()}
        torch.save({"state_dict": model.state_dict(), "n_layers": N_LAYERS, "hidden": HIDDEN, "mean_only": True},
                   OUTPUT_DIR / f"phase2d_{name}.pt")
        print(f"{name}: val EER {fit['best_val_eer']:.4f} at epoch {fit['best_epoch']}, {head_results[name]['train_seconds']:.0f}s, "
              f"{model.n_params()} parameters")
        return model

    models = {}
    for seed in SEEDS:
        models[f"control_s{seed}"] = train_head(f"control_s{seed}", control_rows, seed)
        models[f"F2_s{seed}"] = train_head(f"F2_s{seed}", f2_rows, seed)
        save_progress("heads", head_results)

    # Clean validation scores (for the threshold at the clean-validation EER) and every row's scores on the clean test.
    def score_rows(model, rows):
        return score_model(model, pool_fetch, rows)

    val_clean_labels = pool_labels[val_clean_rows]
    clean_test_rows = np.flatnonzero(~pool_is_aug & np.isin(pool_positions, test_arr))
    clean_test_rows = clean_test_rows[np.argsort(pool_positions[clean_test_rows])]
    clean_test_pos = pool_positions[clean_test_rows]
    clean_test_labels = pool_labels[clean_test_rows]
    for key, model in models.items():
        SEED_SCORES[key] = {"val_clean": score_rows(model, val_clean_rows), "test_clean": score_rows(model, clean_test_rows)}
    del pool, models

## Step 5 -- evaluate every condition

In [ ]:
if FULL:
    ev = load_chunks([CACHE_DIR, *RESUME_DIRS], kinds=("eval",))["eval"]
    assert ev["data"] is not None, "no evaluation features found"
    # Heads are small; reload them from disk so the evaluation does not depend on the training cell's memory.
    heads = {}
    for key in SEED_SCORES:
        blob = torch.load(OUTPUT_DIR / f"phase2d_{key}.pt", map_location="cpu")
        h = SSLHead(blob["n_layers"], blob["hidden"])
        h.load_state_dict(blob["state_dict"])
        heads[key] = h.to(DEVICE).eval()

    def eval_scores(key, rows):
        out = []
        with torch.no_grad():
            for s in range(0, len(rows), 256):
                x = torch.from_numpy(np.asarray(ev["data"][rows[s:s + 256]], dtype=np.float32)).to(DEVICE)
                out.append(heads[key](x).float().cpu())
        return torch.cat(out).numpy().astype(np.float64)

    def mean_over_seeds(model, per_seed):
        return np.mean([per_seed[f"{model}_s{s}"] for s in SEEDS], axis=0)

    def per_seed_eers(model, per_seed, labels):
        return {str(s): compute_eer(per_seed[f"{model}_s{s}"], labels)["eer"] for s in SEEDS}

    # Threshold at the EER on the clean validation clips, per model (score = bonafide logit; lower = more fake).
    thresholds = {m: compute_eer(mean_over_seeds(m, {k: v["val_clean"] for k, v in SEED_SCORES.items()}),
                                 val_clean_labels)["threshold"] for m in ("control", "F2")}

    def evaluate_block(label, per_seed, labels, groups=None, clustered=False):
        d_scores, f_scores = mean_over_seeds("control", per_seed), mean_over_seeds("F2", per_seed)
        block = {"n": {"bonafide": int((labels == BONAFIDE).sum()), "spoof": int((labels == SPOOF).sum()), "rows": int(len(labels))}}
        for name, scores in (("control", d_scores), ("F2", f_scores)):
            ci = bootstrap_eer_ci(scores, labels, n_resamples=N_BOOTSTRAP, seed=SEED)
            block[name] = {"eer": ci["eer"], "ci": [ci["ci_low"], ci["ci_high"]],
                           "per_seed_eer": per_seed_eers(name, per_seed, labels),
                           "genuine_flagged_at_clean_val_eer": share_flagged(scores, labels, thresholds[name])}
        diff = (clustered_paired_bootstrap(f_scores, d_scores, labels, groups, n_resamples=N_BOOTSTRAP, seed=SEED)
                if clustered else
                bootstrap_eer_difference(f_scores, labels, d_scores, labels, n_resamples=N_BOOTSTRAP, seed=SEED, paired=True))
        block["difference_F2_minus_control"] = diff
        block["status"] = criterion_status(diff)
        print(f"{label:22s} control EER {block['control']['eer']:.4f} | F2 EER {block['F2']['eer']:.4f} | "
              f"diff {diff['difference']:+.4f} [{diff['ci_low']:+.4f}, {diff['ci_high']:+.4f}] | "
              f"flagged genuine control {block['control']['genuine_flagged_at_clean_val_eer']:.2f} "
              f"F2 {block['F2']['genuine_flagged_at_clean_val_eer']:.2f}")
        return block

    evaluation = {"thresholds_clean_val_eer": thresholds, "conditions": {}}
    evaluation["clean_full_test"] = evaluate_block("clean (full test)", {k: v["test_clean"] for k, v in SEED_SCORES.items()},
                                                   clean_test_labels)
    pooled = {"scores": {k: [] for k in SEED_SCORES}, "labels": [], "groups": []}
    clean_row_of = {int(p): i for i, p in enumerate(clean_test_pos)}
    for ci_idx, name in enumerate(EVAL_CONDITIONS):
        rows = np.flatnonzero(ev["cond"] == ci_idx)
        rows = rows[np.argsort(ev["positions"][rows])]
        pos = ev["positions"][rows]
        labels = labels_np[pos]
        per_seed = {k: eval_scores(k, rows) for k in SEED_SCORES}
        block = evaluate_block(name, per_seed, labels)
        block["role"] = "training_menu" if name in CONDITIONS["training"] else "held_out_never_trained"
        # The same clips, clean: the baseline each condition is read against.
        take = np.array([clean_row_of[int(p)] for p in pos])
        block["same_clips_clean"] = evaluate_block(f"  {name} (clean, same clips)",
                                                   {k: v["test_clean"][take] for k, v in SEED_SCORES.items()}, labels)
        evaluation["conditions"][name] = block
        if name in CONDITIONS["training"]:
            for k in SEED_SCORES:
                pooled["scores"][k].append(per_seed[k])
            pooled["labels"].append(labels)
            pooled["groups"].append(pos)
    pooled_scores = {k: np.concatenate(v) for k, v in pooled["scores"].items()}
    evaluation["pooled_training_menu_conditions"] = evaluate_block(
        "pooled codec set", pooled_scores, np.concatenate(pooled["labels"]), groups=np.concatenate(pooled["groups"]),
        clustered=True)
    save_progress("evaluation", evaluation)

    # --- the pre-registered criteria, read mechanically --------------------------------------------------------
    c1 = evaluation["clean_full_test"]["status"]["not_worse"]
    per_cond_ok = {n: b["status"]["not_worse"] for n, b in evaluation["conditions"].items() if b["role"] == "training_menu"}
    c2a = all(per_cond_ok.values())
    c2b = evaluation["pooled_training_menu_conditions"]["status"]["better"]
    verdict = {"criterion_1_clean_not_worse": c1, "criterion_2a_no_condition_worse": c2a,
               "criterion_2a_by_condition": per_cond_ok, "criterion_2b_pooled_better": c2b,
               "all_pass": bool(c1 and c2a and c2b),
               "held_out_reported_not_judged": {n: b["difference_F2_minus_control"] for n, b in evaluation["conditions"].items()
                                                if b["role"] != "training_menu"}}
    save_progress("criteria", verdict)
    print("\nPRE-REGISTERED CRITERIA:", json.dumps({k: v for k, v in verdict.items() if k != "held_out_reported_not_judged"}, indent=1))
    print("Held-out conditions are reported, not judged. A failed criterion is a negative result: do not retune and re-run.")

## Wrap-up

In [ ]:
if FULL:
    RESULTS["closing"] = ("Aggregates only; no clip identifiers or per-clip scores are stored. Head weights "
                          f"(phase2d_*.pt, {len(SEED_SCORES)} files) are private until the dataset terms are checked.")
    elapsed = (time.time() - RUN_START) / 3600
    save_progress("elapsed_hours", round(elapsed, 2))
    if DELETE_CACHE_AT_END:
        shutil.rmtree(CACHE_DIR, ignore_errors=True)
        print("feature cache deleted (the run is complete); results and heads are in", OUTPUT_DIR)
    print(f"done in {elapsed:.1f} h. Download results_phase2d.json and the phase2d_*.pt files.")
elif RUN_MODE == "FULL":
    print("Extraction is not finished; see the message above. Nothing was trained.")
else:
    print("Pilot only. Nothing was trained.")